# Full-scale Kaggle-style entity resolution (dual T4)

**Before running:** attach the official `student_resource` dataset, select **GPU T4 x2**, restart the session, then run the notebook from top to bottom. Internet is used only if the small `anyascii` text-normalization dependency is missing from Kaggle's image. Keep `/kaggle/working` intact to resume in the same session. After a session reset, save a notebook version with output files and attach that version as an input; setup looks for a matching checkpoint and restores it when space permits.

This notebook combines the strongest pieces we already have with the broader multi-route candidate search from the team branch. It searches the complete target databases, generates candidates for every test Source 1 row, trains fresh pair matchers, selects a threshold on a clean validation group, checks that choice once on a separate holdout, and writes both required TSV files.

**Model plan:** broad token + character retrieval; pairwise name/address, numeric, cosine, and route-channel features; separate XGBoost GPU matchers for Source 2 and Source 3; LightGBM as a challenger when installed; threshold selection on the challenge's macro F0.5 score. The best validation model or simple XGBoost/LightGBM blend is selected, then scored against all test candidates. There is no global one-to-one assignment. The published first-place Foursquare system also used CatBoost and multilingual transformers; those are follow-up options here because this dataset has only name/address text and those models would add a separate encoder/data pipeline before we know they improve the clean holdout.

**Data scope:** every Source 2 and Source 3 record is searchable for each query. To keep training and storage bounded, this run fits on a deterministic 40% Source 1 query cohort, tunes on 5%, and reports one untouched 5% holdout. That is four times the earlier 10% fit split; all test Source 1 rows are still processed. For fitting only, it keeps every positive pair plus the 16 most text-similar negatives per query. Validation and holdout retain every retrieved pair, so model/threshold selection and reporting use the real candidate distribution. Queries used to choose retrieval settings are excluded from fit, validation, and holdout.

**Expected run time:** highly uncertain. The completed 10k-query broad-retrieval run took about 32.5 minutes at cap 64. A straight query-count extrapolation to roughly 2.8m train-plus-test queries is around **150 hours**, but that is not a reliable prediction because country indexing, cache reuse, concurrency, and feature/output work do not scale linearly. The 128-candidate budget adds downstream work. Use country progress logs to replace this estimate with observed throughput. Kaggle documents a 12-hour CPU/GPU session, so continuing may require saved output checkpoints across sessions; a matching checkpoint restores only when Kaggle exposes it as an attached input and working disk has room.

| Evidence | Reported time | What it covers |
|---|---:|---|
| Measured broad-token retrieval (K32/A16, cap 64) on 10k development queries | 32.5 min; 93.74% candidate-link recall; 97.70% oracle macro F0.5; about 20 GiB parent RSS | Full target search, sampled queries only; not matcher score and not the wider K64/A32, cap-128 configuration below |
| Older retrieval setup on 551,705 training queries | about 68–69 min per shard | Different, less broad configuration; not a safe full-run extrapolation |
| Foursquare 7th-place team | not reported in its writeup | Multiple retrieval routes and a LightGBM pair classifier |
| Foursquare 11th-place team | 24–48 h embedding training; under 1 h for XGBoost | BERT metric-learning system with location features on external compute; not comparable to this text-only run |
| Foursquare 1st-place team | total time not reported | Reported model families/epochs and hardware, but no end-to-end wall time |
| This notebook | ~150 h naive query-count extrapolation; very low confidence | 40% train-query fit cohort plus all test queries, wider retrieval, features, bakeoff, inference, and output assembly |

**Kaggle winner evidence:** the 7th-place Foursquare solution used multilingual text embeddings, multiple candidate routes, and a LightGBM pair classifier; its writeup gives no total elapsed time. The 11th-place team reported 24–48 hours for 40-epoch BERT embedding training on external compute and under one hour for its XGBoost models on Kaggle. That is a different multimodal task with latitude/longitude, and the writeup discusses leakage, so it is only a rough runtime reference. The 1st-place team stacked LightGBM/CatBoost pair models with multilingual transformers and graph post-processing; it also gives no total runtime and explicitly discusses train/test leakage. This notebook transfers only reusable, label-safe retrieval and pairwise-modeling ideas; it does not copy leakage or location features unavailable here.

The official Foursquare writeups are historical examples, not a guarantee that the same models will win this task: [7th-place solution](https://www.kaggle.com/competitions/foursquare-location-matching/writeups/in-tokyo-7th-place-solution-with-inference-code) · [11th-place solution](https://www.kaggle.com/competitions/foursquare-location-matching/writeups/mc-digital-11th-place-code-and-solution) · [1st-place solution](https://www.kaggle.com/competitions/foursquare-location-matching/writeups/re-waiwai-1st-place-solution). Kaggle's current notebook documentation lists a 12-hour CPU/GPU session and 20 GB of auto-saved working disk: [Kaggle notebook limits](https://www.kaggle.com/docs/notebooks).

**Evidence status:** not run here. This workspace has no attached official challenge dataset or Kaggle GPU. The cap-64 development run measured a 97.70% retrieval-oracle macro F0.5 on 10k queries; that candidate set cannot support the 98.9% leaderboard target. The wider cap-128 setup is a measured-quality hypothesis, not a score guarantee. Its clean validation oracle and fitted macro F0.5 are the first decision gates.

In [ ]:
# Kaggle setup: do not replace Kaggle's CUDA-enabled XGBoost build.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ENGINE_DIR = Path("/kaggle/working/amazon_ml_master_engine")
ENGINE_DIR.mkdir(parents=True, exist_ok=True)

retrieval_packages = {
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "anyascii": "anyascii==0.3.3",
}
missing = [package for module, package in retrieval_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing only missing CPU retrieval dependencies:", missing, flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import xgboost as xgb

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    lgb = None
    HAS_LIGHTGBM = False

if torch.cuda.device_count() < 2:
    raise RuntimeError("Select Kaggle GPU T4 x2, then restart and run all cells.")
print({
    "python": sys.version.split()[0], "duckdb": duckdb.__version__,
    "xgboost": xgb.__version__, "torch": torch.__version__,
    "lightgbm_available": HAS_LIGHTGBM,
    "gpus": [torch.cuda.get_device_name(i) for i in range(2)],
}, flush=True)

In [ ]:
%%writefile /kaggle/working/amazon_ml_master_engine/data.py
"""Strict TSV loading and reproducible dataset auditing."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

SOURCE_COLUMNS = ("entity_id", "business_name", "business_address", "country")
TRUTH_COLUMNS = ("source1_entity_id", "matched_entity_ids")


@dataclass(frozen=True, slots=True)
class Record:
    entity_id: str
    business_name: str
    business_address: str
    country: str


def source_path(data_root: Path, split: str, source: int) -> Path:
    if split not in {"train", "test"} or source not in {1, 2, 3}:
        raise ValueError("split must be train/test and source must be 1/2/3")
    return data_root / split / f"{split}_source{source}.tsv"


def iter_records(path: Path, source: int, max_rows: int | None = None) -> Iterator[Record]:
    """Yield records while checking every visited row and ID."""
    seen: set[str] = set()
    if path.suffix == ".gz":
        import gzip
        handle_context = gzip.open(path, "rt", encoding="utf-8-sig", newline="")
    else:
        handle_context = path.open("r", encoding="utf-8-sig", newline="")
    with handle_context as handle:
        reader = csv.reader(handle, delimiter="\t", strict=True)
        try:
            header = tuple(next(reader))
        except StopIteration as exc:
            raise ValueError(f"{path}: empty file") from exc
        if header != SOURCE_COLUMNS:
            raise ValueError(f"{path}: expected header {SOURCE_COLUMNS}, found {header}")
        for line, row in enumerate(reader, start=2):
            if len(row) != 4:
                raise ValueError(f"{path}:{line}: expected 4 tab-separated columns, got {len(row)}")
            entity_id = row[0]
            if not entity_id.startswith(f"S{source}-") or len(entity_id) <= 3:
                raise ValueError(f"{path}:{line}: invalid source-{source} ID {entity_id!r}")
            if entity_id in seen:
                raise ValueError(f"{path}:{line}: duplicate ID {entity_id!r}")
            seen.add(entity_id)
            yield Record(*row)
            if max_rows is not None and len(seen) >= max_rows:
                return


def iter_truth(path: Path) -> Iterator[tuple[str, tuple[str, ...]]]:
    seen: set[str] = set()
    if path.suffix == ".gz":
        import gzip
        handle_context = gzip.open(path, "rt", encoding="utf-8-sig", newline="")
    else:
        handle_context = path.open("r", encoding="utf-8-sig", newline="")
    with handle_context as handle:
        reader = csv.reader(handle, delimiter="\t", strict=True)
        try:
            header = tuple(next(reader))
        except StopIteration as exc:
            raise ValueError(f"{path}: empty file") from exc
        if header != TRUTH_COLUMNS:
            raise ValueError(f"{path}: expected header {TRUTH_COLUMNS}, found {header}")
        for line, row in enumerate(reader, start=2):
            if len(row) != 2:
                raise ValueError(f"{path}:{line}: expected 2 tab-separated columns, got {len(row)}")
            source1_id, ids = row
            if not source1_id.startswith("S1-") or source1_id in seen:
                raise ValueError(f"{path}:{line}: invalid or duplicate Source 1 ID {source1_id!r}")
            seen.add(source1_id)
            matched = tuple(ids.split(",")) if ids else ()
            if len(matched) != len(set(matched)) or any(not mid.startswith(("S2-", "S3-")) for mid in matched):
                raise ValueError(f"{path}:{line}: invalid or repeated target ID")
            yield source1_id, matched


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(4 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def audit_dataset(data_root: Path, include_hashes: bool = True) -> dict:
    """Audit source/label integrity. Uses ID-country maps, not record texts."""
    result: dict = {"files": {}, "truth": {}}
    train_source1_country: dict[str, str] = {}
    train_target_country: dict[str, str] = {}
    for split in ("train", "test"):
        for source in (1, 2, 3):
            path = source_path(data_root, split, source)
            countries: Counter[str] = Counter()
            missing: Counter[str] = Counter()
            lengths = {"business_name": 0, "business_address": 0}
            maximum = {"business_name": 0, "business_address": 0}
            count = 0
            for record in iter_records(path, source):
                count += 1
                countries[record.country] += 1
                for field in ("business_name", "business_address", "country"):
                    value = getattr(record, field)
                    if not value.strip():
                        missing[field] += 1
                    if field in lengths:
                        lengths[field] += len(value)
                        maximum[field] = max(maximum[field], len(value))
                if split == "train":
                    if source == 1:
                        train_source1_country[record.entity_id] = record.country
                    else:
                        train_target_country[record.entity_id] = record.country
            result["files"][str(path.relative_to(data_root))] = {
                "rows": count,
                "countries": dict(sorted(countries.items())),
                "blank_fields": dict(sorted(missing.items())),
                "mean_lengths": {k: round(v / count, 3) if count else 0 for k, v in lengths.items()},
                "max_lengths": maximum,
                "bytes": path.stat().st_size,
                **({"sha256": sha256_file(path)} if include_hashes else {}),
            }
    truth_path = data_root / "train" / "train_ground_truth.tsv"
    truth_ids: set[str] = set()
    cardinality: Counter[int] = Counter()
    invalid_source1: list[str] = []
    invalid_targets: list[str] = []
    cross_country = 0
    total_links = 0
    for source1_id, matches in iter_truth(truth_path):
        truth_ids.add(source1_id)
        cardinality[len(matches)] += 1
        country = train_source1_country.get(source1_id)
        if country is None and len(invalid_source1) < 10:
            invalid_source1.append(source1_id)
        for target_id in matches:
            total_links += 1
            target_country = train_target_country.get(target_id)
            if target_country is None:
                if len(invalid_targets) < 10:
                    invalid_targets.append(target_id)
            elif country is not None and country != target_country:
                cross_country += 1
    result["truth"] = {
        "rows": len(truth_ids),
        "links": total_links,
        "cardinality": {str(k): v for k, v in sorted(cardinality.items())},
        "max_matches": max(cardinality, default=0),
        "missing_source1_label_count": len(train_source1_country.keys() - truth_ids),
        "invalid_source1_examples": invalid_source1,
        "invalid_target_examples": invalid_targets,
        "cross_country_links": cross_country,
        "sha256": sha256_file(truth_path) if include_hashes else None,
    }
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description="Audit the supplied challenge TSVs")
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument("--no-hashes", action="store_true")
    args = parser.parse_args()
    print(json.dumps(audit_dataset(args.data_root, not args.no_hashes), indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/amazon_ml_master_engine/normalization.py
"""Conservative, source-independent text normalization for retrieval."""

from __future__ import annotations

import re
import unicodedata

from anyascii import anyascii

_TOKEN_RE = re.compile(r"[^\w]+", flags=re.UNICODE)
_NAME_VARIANTS = {
    "corp": "corporation",
    "corpn": "corporation",
    "inc": "incorporated",
    "ltd": "limited",
    "pvt": "private",
    "co": "company",
    "intl": "international",
}
_ADDRESS_VARIANTS = {
    "rd": "road",
    "st": "street",
    "ave": "avenue",
    "av": "avenue",
    "blvd": "boulevard",
    "ln": "lane",
    "dr": "drive",
    "hwy": "highway",
    "apt": "apartment",
    "ste": "suite",
    "n": "north",
    "s": "south",
    "e": "east",
    "w": "west",
}
_GENERIC_NAME_TOKENS = frozenset(
    {"the", "and", "company", "corporation", "incorporated", "limited", "private", "llc", "llp"}
)
_GENERIC_ADDRESS_TOKENS = frozenset(
    {"road", "street", "avenue", "boulevard", "lane", "drive", "highway", "apartment",
     "suite", "north", "south", "east", "west", "near", "floor", "unit", "city", "state"}
)


def _tokens(value: str) -> list[str]:
    if not value:
        return []
    # Generic Unicode transliteration is a character normalization rule, not
    # an external business-identity lookup. It helps cross-script name variants.
    value = anyascii(unicodedata.normalize("NFKC", value)).casefold().replace("&", " and ")
    return [token for token in _TOKEN_RE.sub(" ", value).split() if token]


def normalize_name(value: str) -> str:
    """Normalize without deleting legal suffixes or meaningful digits."""
    return " ".join(_NAME_VARIANTS.get(token, token) for token in _tokens(value))


def normalize_address(value: str) -> str:
    return " ".join(_ADDRESS_VARIANTS.get(token, token) for token in _tokens(value))


def informative_name_tokens(normalized_name: str) -> tuple[str, ...]:
    """Return unique tokens eligible for the rare-token retrieval channel."""
    return tuple(sorted({t for t in normalized_name.split() if len(t) >= 3 and t not in _GENERIC_NAME_TOKENS}))


def name_token_signature(normalized_name: str) -> str:
    """Order-independent exact key for at least two informative name words."""
    tokens = informative_name_tokens(normalized_name)
    return " ".join(tokens) if len(tokens) >= 2 else ""


def compact_name(normalized_name: str) -> str:
    """Join informative words to compare a spaced name with a website-style name."""
    words = normalized_name.split()
    if words and words[0] == "www":
        words = words[1:]
    if words and words[-1] in {"com", "net", "org", "in"}:
        words = words[:-1]
    return "".join(word for word in words if len(word) >= 3 and word not in _GENERIC_NAME_TOKENS)


def informative_address_tokens(normalized_address: str) -> tuple[str, ...]:
    """Keep distinctive words and multi-digit building/PIN numbers."""
    return tuple(sorted({t for t in normalized_address.split()
                         if len(t) >= 3 and t not in _GENERIC_ADDRESS_TOKENS}))

In [ ]:
%%writefile /kaggle/working/amazon_ml_master_engine/candidates.py
"""Bounded candidate retrieval for the Amazon ML entity-resolution challenge.

The long-form output is the exact pair set Person B must score. Intermediate
per-source/chunk files make the expensive retrieval resumable and permit cap
experiments without recomputing similarities. No external identity data is used.
"""

from __future__ import annotations

import argparse
import csv
import gzip
import json
import math
import multiprocessing
import os
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer

try:  # permit both `python -m src.candidates` and direct script execution
    from .data import Record, iter_records, iter_truth, source_path
    from .normalization import (compact_name, informative_address_tokens, informative_name_tokens,
                                name_token_signature, normalize_address, normalize_name)
except ImportError:
    from data import Record, iter_records, iter_truth, source_path
    from normalization import (compact_name, informative_address_tokens, informative_name_tokens,
                               name_token_signature, normalize_address, normalize_name)

HEADER = ("source1_entity_id", "candidate_entity_id", "name_cosine", "address_cosine", "retrieval_channels")
CHANNEL_ORDER = ("exact_name", "exact_address", "name_signature", "compact_name", "rare_name", "rare_address", "name_pair", "address_pair", "cross_token", "single_name", "single_address", "name_char", "address_char")
RETRIEVAL_VERSION = "name-key-1"


@dataclass(frozen=True)
class Settings:
    name_k: int = 32
    address_k: int = 16
    cap: int = 32
    stage_cap: int = 64
    query_chunk: int = 2048
    matrix_chunk: int = 100_000
    hash_features: int = 1 << 20
    max_common_df: float = 0.05
    rare_max_df: int = 64
    rare_address_max_df: int = 128
    max_exact_block: int = 2_000
    indexed_grams: int = 8
    query_grams: int = 16
    gram_selection: str = "rarest"
    min_index_df: int = 1
    max_probe_df: int = 2_000
    cross_max_df: int = 1_000
    cross_name_tokens: int = 3
    cross_address_tokens: int = 4
    pair_max_df: int = 1_000
    pair_max_hits: int = 0
    single_max_df: int = 0
    single_tokens: int = 0
    name_keys: bool = False
    workers: int = 1


@dataclass
class FieldIndex:
    vectorizer: HashingVectorizer
    tfidf: TfidfTransformer
    target: sparse.csr_matrix
    common: np.ndarray
    posting_offsets: np.ndarray
    posting_rows: np.ndarray

    def query(self, texts: list[str]) -> sparse.csr_matrix:
        matrix = self.vectorizer.transform(texts).tocsr()
        matrix.data[self.common[matrix.indices]] = 0
        matrix.eliminate_zeros()
        return self.tfidf.transform(matrix).tocsr()


@dataclass
class TokenIndex:
    """Compact postings for informative whole tokens, including moderate-frequency ones."""

    vectorizer: HashingVectorizer
    posting_offsets: np.ndarray
    posting_rows: np.ndarray

    def probe(self, query_row: sparse.csr_matrix, limit: int, max_df: int) -> np.ndarray:
        features = query_row.indices
        if not len(features) or limit <= 0:
            return np.empty(0, dtype=np.int32)
        sizes = self.posting_offsets[features + 1] - self.posting_offsets[features]
        eligible = np.flatnonzero((sizes > 0) & (sizes <= max_df))
        if not len(eligible):
            return np.empty(0, dtype=np.int32)
        order = np.lexsort((features[eligible], sizes[eligible]))[:limit]
        postings = [self.posting_rows[self.posting_offsets[features[position]]:
                                     self.posting_offsets[features[position] + 1]]
                    for position in eligible[order]]
        return np.unique(np.concatenate(postings))

    def pair_probe(self, query_row: sparse.csr_matrix, limit: int, max_df: int,
                   max_hits: int = 0) -> np.ndarray:
        """Find targets sharing two informative query words in this field."""
        features = query_row.indices
        if len(features) < 2 or limit < 2:
            return np.empty(0, dtype=np.int32)
        sizes = self.posting_offsets[features + 1] - self.posting_offsets[features]
        eligible = np.flatnonzero((sizes > 0) & (sizes <= max_df))
        if len(eligible) < 2:
            return np.empty(0, dtype=np.int32)
        positions = eligible[np.lexsort((features[eligible], sizes[eligible]))[:limit]]
        postings = [self.posting_rows[self.posting_offsets[features[position]]:
                                     self.posting_offsets[features[position] + 1]]
                    for position in positions]
        intersections = [np.intersect1d(postings[i], postings[j], assume_unique=True)
                         for i in range(len(postings)) for j in range(i + 1, len(postings))]
        hits = [rows for rows in intersections if len(rows) and (not max_hits or len(rows) <= max_hits)]
        return np.unique(np.concatenate(hits)) if hits else np.empty(0, dtype=np.int32)


def _token_index(texts: list[str], settings: Settings, informative_tokens) -> TokenIndex:
    vectorizer = HashingVectorizer(
        analyzer="word", token_pattern=r"(?u)\b\w+\b", n_features=settings.hash_features,
        alternate_sign=False, norm=None, binary=True, dtype=np.float32,
    )
    blocks = []
    for start in range(0, len(texts), settings.matrix_chunk):
        normalized = [" ".join(informative_tokens(text))
                      for text in texts[start:start + settings.matrix_chunk]]
        blocks.append(vectorizer.transform(normalized))
    matrix = sparse.vstack(blocks, format="csr", dtype=np.float32)
    del blocks
    postings = matrix.tocsc()
    return TokenIndex(vectorizer, postings.indptr, postings.indices)


def _field_index(texts: list[str], settings: Settings) -> FieldIndex:
    vectorizer = HashingVectorizer(
        analyzer="char", ngram_range=(3, 5), n_features=settings.hash_features,
        alternate_sign=False, norm=None, dtype=np.float32,
    )
    blocks = [vectorizer.transform(texts[i:i + settings.matrix_chunk])
              for i in range(0, len(texts), settings.matrix_chunk)]
    matrix = sparse.vstack(blocks, format="csr", dtype=np.float32) if blocks else sparse.csr_matrix((0, settings.hash_features), dtype=np.float32)
    del blocks
    if matrix.shape[0] == 0:
        raise ValueError("target source contains no records")
    # Frequent character grams create enormous postings and little discrimination.
    # CSR has one entry per (record, hashed feature), so bincount gives document frequency.
    df = np.bincount(matrix.indices, minlength=settings.hash_features)
    common = df > max(2, int(settings.max_common_df * len(texts)))
    if common.any():
        matrix.data[common[matrix.indices]] = 0
        matrix.eliminate_zeros()
    tfidf = TfidfTransformer(norm="l2", use_idf=True, sublinear_tf=True)
    matrix = tfidf.fit_transform(matrix).tocsr().astype(np.float32)
    # Index only each target's rarest grams. A dense all-pairs sparse product
    # touches far too many common-character postings at challenge scale.
    capacity = len(texts) * settings.indexed_grams
    indexed_features = np.empty(capacity, dtype=np.int32)
    indexed_rows = np.empty(capacity, dtype=np.int32)
    cursor = 0
    for row in range(matrix.shape[0]):
        start, stop = matrix.indptr[row], matrix.indptr[row + 1]
        features = matrix.indices[start:stop]
        if not len(features):
            continue
        eligible = features[(df[features] >= settings.min_index_df) &
                            (df[features] <= settings.max_probe_df)]
        if not len(eligible):
            continue
        count = min(len(eligible), settings.indexed_grams)
        if settings.gram_selection == "hash":
            # Stable bottom-k feature sampling avoids filling an index row
            # solely with typo-induced, document-frequency-one grams.
            priorities = eligible.astype(np.uint64) * np.uint64(11400714819323198485)
        else:
            priorities = df[eligible]
        selected = eligible[np.argpartition(priorities, count - 1)[:count]]
        selected.sort()
        indexed_features[cursor:cursor + count] = selected
        indexed_rows[cursor:cursor + count] = row
        cursor += count
    indexed_features = indexed_features[:cursor]
    indexed_rows = indexed_rows[:cursor]
    counts = np.bincount(indexed_features, minlength=settings.hash_features)
    offsets = np.empty(settings.hash_features + 1, dtype=np.int64)
    offsets[0] = 0
    np.cumsum(counts, out=offsets[1:])
    order = np.argsort(indexed_features, kind="stable")
    posting_rows = indexed_rows[order]
    return FieldIndex(vectorizer, tfidf, matrix, common, offsets, posting_rows)


def _best_field_hits(index: FieldIndex, query_row: sparse.csr_matrix,
                     settings: Settings, k: int) -> np.ndarray:
    features = query_row.indices
    if not len(features) or k <= 0:
        return np.empty(0, dtype=np.int32)
    sizes = index.posting_offsets[features + 1] - index.posting_offsets[features]
    eligible = np.flatnonzero((sizes > 0) & (sizes <= settings.max_probe_df))
    if not len(eligible):
        return np.empty(0, dtype=np.int32)
    order = np.lexsort((features[eligible], sizes[eligible]))[:settings.query_grams]
    postings = []
    weights = []
    for position in eligible[order]:
        feature = features[position]
        start, stop = index.posting_offsets[feature:feature + 2]
        postings.append(index.posting_rows[start:stop])
        weights.append(np.full(stop - start, query_row.data[position], dtype=np.float32))
    rows, inverse = np.unique(np.concatenate(postings), return_inverse=True)
    votes = np.bincount(inverse, weights=np.concatenate(weights))
    ranked = np.lexsort((rows, -votes))[:k]
    return rows[ranked]


def _rank(name: float, address: float, channels: str) -> float:
    selected = set(channels.split("|"))
    return (0.55 * name + 0.45 * address + 0.12 * ("exact_name" in selected)
            + 0.08 * ("exact_address" in selected) + 0.08 * ("name_signature" in selected)
            + 0.04 * ("compact_name" in selected) + 0.02 * ("rare_name" in selected)
            + 0.02 * ("rare_address" in selected))


def _format_score(value: float) -> str:
    value = min(1.0, max(0.0, float(value)))
    if not math.isfinite(value):
        raise ValueError("non-finite cosine similarity")
    return f"{value:.6f}"


def _memory_state() -> dict[str, float]:
    """Report parent-process and, when available, container-wide memory."""
    state: dict[str, float] = {}
    try:
        import psutil
        info = psutil.Process().memory_info()
        peak = getattr(info, "peak_wset", info.rss)
        if os.name == "posix":
            import resource
            # ru_maxrss is KiB on Linux and bytes on macOS.
            peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            if os.uname().sysname == "Linux":
                peak *= 1024
        state.update({"rss_gb": round(info.rss / (1 << 30), 3),
                      "peak_rss_gb": round(peak / (1 << 30), 3)})
    except ImportError:
        pass
    for key, paths in {
        "container_current_gb": ("/sys/fs/cgroup/memory.current",
                                 "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
        "container_peak_gb": ("/sys/fs/cgroup/memory.peak",
                              "/sys/fs/cgroup/memory/memory.max_usage_in_bytes"),
    }.items():
        for path in paths:
            try:
                value = int(Path(path).read_text(encoding="ascii").strip())
            except (OSError, ValueError):
                continue
            state[key] = round(value / (1 << 30), 3)
            break
    return state


def _load_sorted_source1(path: Path, limit: int | None) -> list[Record]:
    return sorted(iter_records(path, 1, max_rows=limit), key=lambda record: record.entity_id)


def _select_source1(source1: list[Record], sample_split: str | None,
                    shard_index: int = 0, shard_count: int = 1) -> list[Record]:
    """Select seeded evaluation rows or one sorted contiguous final-run shard."""
    if sample_split is None:
        start = len(source1) * shard_index // shard_count
        stop = len(source1) * (shard_index + 1) // shard_count
        return source1[start:stop]
    indices = random.Random(20260926).sample(range(len(source1)), min(60_000, len(source1)))
    chosen = indices[:10_000] if sample_split == "dev" else indices[10_000:60_000]
    return [source1[index] for index in sorted(chosen)]


def _build_lookups(names: list[str], addresses: list[str], settings: Settings):
    exact_names: dict[str, list[int]] = defaultdict(list)
    exact_addresses: dict[str, list[int]] = defaultdict(list)
    name_signatures: dict[str, list[int]] = defaultdict(list)
    compact_names: dict[str, list[int]] = defaultdict(list)
    frequencies: Counter[str] = Counter()
    address_frequencies: Counter[str] = Counter()
    for idx, (name, address) in enumerate(zip(names, addresses, strict=True)):
        if name:
            exact_names[name].append(idx)
            frequencies.update(informative_name_tokens(name))
            if settings.name_keys:
                signature = name_token_signature(name)
                if signature:
                    name_signatures[signature].append(idx)
                compact = compact_name(name)
                if len(compact) >= 8 and (len(name.split()) == 1 or name.startswith("www ") or
                                           name.endswith((" com", " net", " org", " in"))):
                    compact_names[compact].append(idx)
        if address:
            exact_addresses[address].append(idx)
            address_frequencies.update(informative_address_tokens(address))
    rare: dict[str, list[int]] = defaultdict(list)
    rare_address: dict[str, list[int]] = defaultdict(list)
    for idx, name in enumerate(names):
        for token in informative_name_tokens(name):
            if frequencies[token] <= settings.rare_max_df:
                rare[token].append(idx)
    for idx, address in enumerate(addresses):
        for token in informative_address_tokens(address):
            if address_frequencies[token] <= settings.rare_address_max_df:
                rare_address[token].append(idx)
    return (exact_names, exact_addresses, name_signatures, compact_names,
            rare, frequencies, rare_address, address_frequencies)


def _part_path(work_dir: Path, source: int, chunk_number: int) -> Path:
    return work_dir / f"source{source}" / f"part_{chunk_number:06d}.tsv.gz"


def _read_part(path: Path) -> dict[str, list[tuple[str, str, str, str, str]]]:
    groups: dict[str, list[tuple[str, str, str, str, str]]] = defaultdict(list)
    with gzip.open(path, "rt", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle, delimiter="\t")
        if tuple(next(reader)) != HEADER:
            raise ValueError(f"{path}: invalid staged header")
        for row in reader:
            if len(row) != 5:
                raise ValueError(f"{path}: invalid staged row")
            groups[row[0]].append(tuple(row))
    return groups


@dataclass
class _RetrievalState:
    source1: list[Record]
    target_ids: list[str]
    exact_names: dict[str, list[int]]
    exact_addresses: dict[str, list[int]]
    name_signatures: dict[str, list[int]]
    compact_names: dict[str, list[int]]
    rare_names: dict[str, list[int]]
    name_frequencies: Counter[str]
    rare_addresses: dict[str, list[int]]
    address_frequencies: Counter[str]
    name_index: FieldIndex
    address_index: FieldIndex
    name_tokens: TokenIndex
    address_tokens: TokenIndex
    settings: Settings
    work_dir: Path
    source: int


_FORK_STATE: _RetrievalState | None = None


def _run_chunk(chunk_number: int) -> int:
    state = _FORK_STATE
    if state is None:
        raise RuntimeError("retrieval state was not initialized before worker fork")
    settings = state.settings
    batch = state.source1[chunk_number * settings.query_chunk:(chunk_number + 1) * settings.query_chunk]
    query_names = [normalize_name(record.business_name) for record in batch]
    query_addresses = [normalize_address(record.business_address) for record in batch]
    name_query = state.name_index.query(query_names)
    address_query = state.address_index.query(query_addresses)
    name_token_query = state.name_tokens.vectorizer.transform(
        [" ".join(informative_name_tokens(name)) for name in query_names]).tocsr()
    address_token_query = state.address_tokens.vectorizer.transform(
        [" ".join(informative_address_tokens(address)) for address in query_addresses]).tocsr()
    part = _part_path(state.work_dir, state.source, chunk_number)
    part.parent.mkdir(parents=True, exist_ok=True)
    temp = part.with_suffix(part.suffix + ".tmp")
    with gzip.open(temp, "wt", encoding="utf-8", newline="", compresslevel=3) as handle:
        writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
        writer.writerow(HEADER)
        for row_number, record in enumerate(batch):
            found: dict[int, set[str]] = defaultdict(set)
            name = query_names[row_number]
            address = query_addresses[row_number]
            if name and len(state.exact_names.get(name, ())) <= settings.max_exact_block:
                for idx in state.exact_names.get(name, ()):
                    found[idx].add("exact_name")
            if address and len(state.exact_addresses.get(address, ())) <= settings.max_exact_block:
                for idx in state.exact_addresses.get(address, ()):
                    found[idx].add("exact_address")
            if settings.name_keys and name:
                signature = name_token_signature(name)
                if signature and len(state.name_signatures.get(signature, ())) <= settings.max_exact_block:
                    for idx in state.name_signatures.get(signature, ()):
                        found[idx].add("name_signature")
                compact = compact_name(name)
                if len(compact) >= 8 and len(state.compact_names.get(compact, ())) <= settings.max_exact_block:
                    for idx in state.compact_names.get(compact, ()):
                        found[idx].add("compact_name")
            tokens = informative_name_tokens(name)
            for token in sorted((t for t in tokens if t in state.rare_names),
                                key=lambda t: (state.name_frequencies[t], t))[:3]:
                for idx in state.rare_names[token]:
                    found[idx].add("rare_name")
            address_tokens = informative_address_tokens(address)
            for token in sorted((t for t in address_tokens if t in state.rare_addresses),
                                key=lambda t: (state.address_frequencies[t], t))[:4]:
                for idx in state.rare_addresses[token]:
                    found[idx].add("rare_address")
            cross_name = state.name_tokens.probe(name_token_query[row_number],
                                                 settings.cross_name_tokens, settings.cross_max_df)
            cross_address = state.address_tokens.probe(address_token_query[row_number],
                                                       settings.cross_address_tokens, settings.cross_max_df)
            for idx in state.name_tokens.pair_probe(name_token_query[row_number],
                                                    settings.cross_name_tokens, settings.pair_max_df,
                                                    settings.pair_max_hits):
                found[int(idx)].add("name_pair")
            for idx in state.address_tokens.pair_probe(address_token_query[row_number],
                                                       settings.cross_address_tokens, settings.pair_max_df,
                                                       settings.pair_max_hits):
                found[int(idx)].add("address_pair")
            if len(cross_name) and len(cross_address):
                for idx in np.intersect1d(cross_name, cross_address, assume_unique=True):
                    found[int(idx)].add("cross_token")
            if settings.single_tokens and settings.single_max_df:
                for idx in state.name_tokens.probe(name_token_query[row_number],
                                                   settings.single_tokens, settings.single_max_df):
                    found[int(idx)].add("single_name")
                for idx in state.address_tokens.probe(address_token_query[row_number],
                                                      settings.single_tokens, settings.single_max_df):
                    found[int(idx)].add("single_address")
            for idx in _best_field_hits(state.name_index, name_query[row_number], settings, settings.name_k):
                found[int(idx)].add("name_char")
            for idx in _best_field_hits(state.address_index, address_query[row_number], settings, settings.address_k):
                found[int(idx)].add("address_char")
            if not found:
                continue
            indices = np.fromiter(found, dtype=np.int32)
            name_scores = state.name_index.target[indices].dot(name_query[row_number].T).toarray().ravel()
            address_scores = state.address_index.target[indices].dot(address_query[row_number].T).toarray().ravel()
            candidates = []
            for pos, idx in enumerate(indices):
                channels = "|".join(channel for channel in CHANNEL_ORDER if channel in found[int(idx)])
                nscore = float(name_scores[pos])
                ascore = float(address_scores[pos])
                candidates.append((state.target_ids[int(idx)], _format_score(nscore), _format_score(ascore), channels,
                                   _rank(nscore, ascore, channels)))
            # Keep the proposal shortlist separate from the final candidate cap.
            candidates.sort(key=lambda row: (-row[4], row[0]))
            candidates = sorted(candidates[:max(settings.stage_cap, settings.name_k + settings.address_k)], key=lambda row: row[0])
            for target_id, nscore, ascore, channels, _ in candidates:
                writer.writerow((record.entity_id, target_id, nscore, ascore, channels))
    os.replace(temp, part)
    return chunk_number


def _retrieve_source(
    source1: list[Record], target_path: Path, source: int, settings: Settings,
    work_dir: Path, limit_targets: int | None,
) -> None:
    global _FORK_STATE
    total_chunks = (len(source1) + settings.query_chunk - 1) // settings.query_chunk
    missing = [i for i in range(total_chunks) if not _part_path(work_dir, source, i).is_file()]
    if not missing:
        return
    started = time.monotonic()
    targets = list(iter_records(target_path, source, max_rows=limit_targets))
    target_ids = [record.entity_id for record in targets]
    names = [normalize_name(record.business_name) for record in targets]
    addresses = [normalize_address(record.business_address) for record in targets]
    (exact_names, exact_addresses, name_signatures, compact_names,
     rare, frequencies, rare_address, address_frequencies) = _build_lookups(names, addresses, settings)
    name_index = _field_index(names, settings)
    address_index = _field_index(addresses, settings)
    name_tokens = _token_index(names, settings, informative_name_tokens)
    address_tokens = _token_index(addresses, settings, informative_address_tokens)
    print(json.dumps({"stage": "target_index", "source": source, "targets": len(targets),
                      "seconds": round(time.monotonic() - started, 2), **_memory_state()}), flush=True)
    del targets, names, addresses
    _FORK_STATE = _RetrievalState(source1, target_ids, exact_names, exact_addresses,
                                  name_signatures, compact_names, rare,
                                  frequencies, rare_address, address_frequencies,
                                  name_index, address_index, name_tokens, address_tokens,
                                  settings, work_dir, source)
    if settings.workers > 1 and os.name == "posix":
        with multiprocessing.get_context("fork").Pool(processes=settings.workers) as pool:
            completed = pool.imap_unordered(_run_chunk, missing, chunksize=1)
            for count, chunk_number in enumerate(completed, start=1):
                if count == 1 or count % 20 == 0 or count == len(missing):
                    print(json.dumps({"stage": "retrieval", "source": source, "completed_chunks": count,
                                      "of": total_chunks, "last_chunk": chunk_number + 1,
                                      "seconds": round(time.monotonic() - started, 2),
                                      **_memory_state()}), flush=True)
    else:
        if settings.workers > 1:
            print(json.dumps({"stage": "warning", "message": "fork workers unavailable; using one process"}), flush=True)
        for count, chunk_number in enumerate(missing, start=1):
            _run_chunk(chunk_number)
            if count == 1 or count % 20 == 0 or count == len(missing):
                print(json.dumps({"stage": "retrieval", "source": source, "completed_chunks": count,
                                  "of": total_chunks, "last_chunk": chunk_number + 1,
                                  "seconds": round(time.monotonic() - started, 2),
                                  **_memory_state()}), flush=True)
    _FORK_STATE = None


def _choose_rows(rows2: list, rows3: list, cap: int) -> list:
    def ordered(rows):
        return sorted(rows, key=lambda row: (-_rank(float(row[2]), float(row[3]), row[4]), row[1]))
    left, right = ordered(rows2), ordered(rows3)
    chosen = []
    quota = cap // 2
    chosen.extend(left[:quota])
    chosen.extend(right[:quota])
    remainder = left[quota:] + right[quota:]
    remainder.sort(key=lambda row: (-_rank(float(row[2]), float(row[3]), row[4]), row[1]))
    chosen.extend(remainder[:cap - len(chosen)])
    return sorted(chosen, key=lambda row: row[1])


def _manifest(data_root: Path, split: str, settings: Settings, limit_source1: int | None,
              limit_targets: int | None, sample_split: str | None,
              shard_index: int, shard_count: int) -> dict:
    paths = [source_path(data_root, split, source) for source in (1, 2, 3)]
    retrieval_settings = {key: value for key, value in vars(settings).items() if key not in {"cap", "workers"}}
    return {"retrieval_version": RETRIEVAL_VERSION, "split": split, "settings": retrieval_settings,
            "limit_source1": limit_source1, "sample_split": sample_split,
            "shard_index": shard_index, "shard_count": shard_count,
            "limit_targets": limit_targets, "inputs": {str(p.resolve()): [p.stat().st_size, p.stat().st_mtime_ns] for p in paths}}


def evaluate_retrieval(candidate_path: Path, truth_path: Path, selected_ids: set[str] | None = None) -> dict:
    """Measure candidate coverage and its macro-F0.5 ceiling on train rows.

    This is an oracle ceiling, not the score of a trained matcher. It requires
    candidate rows for every selected Source 1 record, including implicit
    zero-candidate rows. Do not use on `--limit-*` benchmark output.
    """
    truth = {source1: set(matches) for source1, matches in iter_truth(truth_path)
             if selected_ids is None or source1 in selected_ids}
    if selected_ids is not None and len(truth) != len(selected_ids):
        raise ValueError("selected Source 1 IDs absent from ground truth")
    total_truth = len(truth)
    total_links = sum(len(matches) for matches in truth.values())
    all_counts: list[int] = []
    covered_links = 0
    fully_covered_all = 0
    fully_covered_nonempty = 0
    nonempty_truth = sum(bool(matches) for matches in truth.values())
    oracle_score = 0.0
    total_pairs = 0
    last_pair = ("", "")

    def finish(source1: str, candidates: set[str]) -> None:
        nonlocal covered_links, fully_covered_all, fully_covered_nonempty, oracle_score
        matches = truth.pop(source1, None)
        if matches is None:
            raise ValueError(f"candidate Source 1 ID absent from ground truth: {source1}")
        all_counts.append(len(candidates))
        hits = len(matches & candidates)
        covered_links += hits
        complete = hits == len(matches)
        fully_covered_all += complete
        fully_covered_nonempty += complete and bool(matches)
        if not matches:
            oracle_score += 1.0
        elif hits:
            recall = hits / len(matches)
            oracle_score += 1.25 * recall / (0.25 + recall)

    current_id = ""
    candidates: set[str] = set()
    with candidate_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle, delimiter="\t")
        if tuple(next(reader)) != HEADER:
            raise ValueError("candidate file has an invalid header")
        for row in reader:
            if len(row) != 5:
                raise ValueError("candidate file has an invalid row")
            source1, target = row[0], row[1]
            if (source1, target) <= last_pair:
                raise ValueError("candidate pairs are not unique and sorted")
            last_pair = (source1, target)
            if current_id and source1 != current_id:
                finish(current_id, candidates)
                candidates = set()
            current_id = source1
            candidates.add(target)
            total_pairs += 1
    if current_id:
        finish(current_id, candidates)
    for source1, matches in truth.items():
        all_counts.append(0)
        fully_covered_all += not matches
        oracle_score += not matches
    counts = np.asarray(all_counts, dtype=np.int32)
    return {"source1_rows": total_truth, "true_links": total_links, "candidate_pairs": total_pairs,
            "true_link_recall": covered_links / total_links if total_links else 1.0,
            "complete_set_coverage_all": fully_covered_all / total_truth if total_truth else 1.0,
            "complete_set_coverage_nonempty": fully_covered_nonempty / nonempty_truth if nonempty_truth else 1.0,
            "oracle_macro_f0_5": oracle_score / total_truth if total_truth else 1.0,
            "candidate_mean": float(counts.mean()) if len(counts) else 0.0,
            "candidate_p95": float(np.percentile(counts, 95)) if len(counts) else 0.0,
            "zero_candidate_rows": int(np.sum(counts == 0))}


def generate(data_root: Path, split: str, out: Path, work_dir: Path, settings: Settings,
             limit_source1: int | None = None, limit_targets: int | None = None,
             sample_split: str | None = None, shard_index: int = 0,
             shard_count: int = 1) -> set[str] | None:
    if (settings.cap < 1 or settings.name_k < 0 or settings.address_k < 0 or
            settings.stage_cap < 1 or settings.query_chunk < 1 or settings.workers < 1 or
            settings.indexed_grams < 1 or settings.query_grams < 1 or
            settings.min_index_df < 1 or settings.max_probe_df < 1):
        raise ValueError("cap/chunk/workers/gram budgets must be positive; top-K values must be nonnegative")
    if settings.gram_selection not in {"rarest", "hash"}:
        raise ValueError("gram_selection must be rarest or hash")
    if (settings.cross_max_df < 1 or settings.pair_max_df < 1 or settings.pair_max_hits < 0 or
            settings.single_max_df < 0 or settings.single_tokens < 0 or
            settings.cross_name_tokens < 0 or settings.cross_address_tokens < 0):
        raise ValueError("token frequency must be positive; optional token budgets must be nonnegative")
    if shard_count < 1 or not 0 <= shard_index < shard_count:
        raise ValueError("shard_count must be positive and 0 <= shard_index < shard_count")
    if sample_split and shard_count != 1:
        raise ValueError("sample split and final-run sharding cannot be combined")
    work_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = work_dir / "manifest.json"
    expected = _manifest(data_root, split, settings, limit_source1, limit_targets,
                         sample_split, shard_index, shard_count)
    if manifest_path.exists():
        found = json.loads(manifest_path.read_text(encoding="utf-8"))
        if found != expected:
            raise ValueError(f"{work_dir}: staged data has different inputs or settings; choose a fresh --work-dir")
    else:
        if any(work_dir.iterdir()):
            raise ValueError(f"{work_dir}: nonempty work directory has no manifest")
        manifest_path.write_text(json.dumps(expected, indent=2), encoding="utf-8")
    source1 = _select_source1(_load_sorted_source1(source_path(data_root, split, 1), limit_source1),
                              sample_split, shard_index, shard_count)
    for source in (2, 3):
        _retrieve_source(source1, source_path(data_root, split, source), source, settings, work_dir, limit_targets)
    out.parent.mkdir(parents=True, exist_ok=True)
    temp_out = out.with_suffix(out.suffix + ".tmp")
    emitted = 0
    with temp_out.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
        writer.writerow(HEADER)
        for chunk_number, start in enumerate(range(0, len(source1), settings.query_chunk)):
            groups2 = _read_part(_part_path(work_dir, 2, chunk_number))
            groups3 = _read_part(_part_path(work_dir, 3, chunk_number))
            for record in source1[start:start + settings.query_chunk]:
                rows = _choose_rows(groups2.get(record.entity_id, []), groups3.get(record.entity_id, []), settings.cap)
                for row in rows:
                    writer.writerow(row)
                    emitted += 1
    os.replace(temp_out, out)
    print(json.dumps({"stage": "complete", "source1": len(source1), "pairs": emitted,
                      "output": str(out), "work_dir": str(work_dir)}), flush=True)
    return {record.entity_id for record in source1} if sample_split or shard_count > 1 else None


def main() -> None:
    parser = argparse.ArgumentParser(description="Generate the final long-form candidate set")
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument("--split", choices=("train", "test"), required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--work-dir", type=Path, help="Persistent, private checkpoint directory")
    parser.add_argument("--name-k", type=int, default=32)
    parser.add_argument("--address-k", type=int, default=16)
    parser.add_argument("--cap", type=int, default=32)
    parser.add_argument("--stage-cap", type=int, default=64,
                        help="Maximum ranked proposals staged per target source before the final cap")
    parser.add_argument("--query-chunk", type=int, default=2048)
    parser.add_argument("--matrix-chunk", type=int, default=100_000)
    parser.add_argument("--hash-features", type=int, default=1 << 20)
    parser.add_argument("--indexed-grams", type=int, default=8,
                        help="Maximum rare character grams posted per target field")
    parser.add_argument("--query-grams", type=int, default=16,
                        help="Maximum rare character grams probed per query field")
    parser.add_argument("--gram-selection", choices=("rarest", "hash"), default="rarest",
                        help="Choose target gram postings by rarest document frequency or stable hash")
    parser.add_argument("--min-index-df", type=int, default=1,
                        help="Minimum target frequency of indexed character grams")
    parser.add_argument("--max-probe-df", type=int, default=2_000,
                        help="Maximum target frequency of each probed character gram")
    parser.add_argument("--pair-max-df", type=int, default=1_000,
                        help="Maximum target frequency of a word used in same-field pair retrieval")
    parser.add_argument("--cross-max-df", type=int, default=1_000,
                        help="Maximum word frequency used in cross-field intersection proposals")
    parser.add_argument("--cross-name-tokens", type=int, default=3,
                        help="Informative query name words available to pair and cross-field probes")
    parser.add_argument("--cross-address-tokens", type=int, default=4,
                        help="Informative query address words available to pair and cross-field probes")
    parser.add_argument("--pair-max-hits", type=int, default=0,
                        help="Skip word-pair blocks larger than this; zero disables this limit")
    parser.add_argument("--single-max-df", type=int, default=0,
                        help="Maximum target frequency for single-word proposals; zero disables them")
    parser.add_argument("--single-tokens", type=int, default=0,
                        help="Rare query words to probe per field for single-word proposals")
    parser.add_argument("--name-keys", action="store_true",
                        help="Try word-order-independent and compact website-style name keys")
    parser.add_argument("--workers", type=int, default=1, help="POSIX fork workers sharing the target index")
    parser.add_argument("--limit-source1", type=int, help="Benchmark only; not valid for final output")
    parser.add_argument("--limit-targets", type=int, help="Benchmark only; not valid for final output")
    parser.add_argument("--sample-split", choices=("dev", "holdout"),
                        help="Seeded disjoint 10k development or 50k holdout rows; train only")
    parser.add_argument("--shard-index", type=int, default=0, help="Zero-based sorted Source 1 shard")
    parser.add_argument("--shard-count", type=int, default=1, help="Number of contiguous Source 1 shards")
    parser.add_argument("--report", type=Path, help="Write train retrieval metrics after full generation")
    args = parser.parse_args()
    if args.sample_split and (args.split != "train" or args.limit_source1 is not None):
        parser.error("--sample-split requires train and cannot combine with --limit-source1")
    if args.report and (args.split != "train" or args.limit_source1 is not None or args.limit_targets is not None):
        parser.error("--report requires complete target sources and no --limit-* flags")
    settings = Settings(name_k=args.name_k, address_k=args.address_k, cap=args.cap,
                        stage_cap=args.stage_cap,
                        query_chunk=args.query_chunk, matrix_chunk=args.matrix_chunk,
                        hash_features=args.hash_features, indexed_grams=args.indexed_grams,
                        query_grams=args.query_grams, gram_selection=args.gram_selection,
                        min_index_df=args.min_index_df, max_probe_df=args.max_probe_df,
                        pair_max_df=args.pair_max_df, pair_max_hits=args.pair_max_hits,
                        cross_max_df=args.cross_max_df,
                        cross_name_tokens=args.cross_name_tokens,
                        cross_address_tokens=args.cross_address_tokens,
                        single_max_df=args.single_max_df, single_tokens=args.single_tokens,
                        name_keys=args.name_keys,
                        workers=args.workers)
    work_dir = args.work_dir or args.out.parent / f"{args.out.stem}.work"
    selected_ids = generate(args.data_root, args.split, args.out, work_dir, settings,
                            args.limit_source1, args.limit_targets, args.sample_split,
                            args.shard_index, args.shard_count)
    if args.report:
        report = evaluate_retrieval(args.out, args.data_root / "train" / "train_ground_truth.tsv", selected_ids)
        args.report.parent.mkdir(parents=True, exist_ok=True)
        args.report.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print(json.dumps({"stage": "evaluation", **report}), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/amazon_ml_master_engine/output_assembly.py
"""Memory-bounded submission assembly and recovery from completed Parquet files.

This module is embedded into both notebooks by sync_notebook.py. It imports no
GPU framework and never trains a model or changes the accepted candidate set.
"""

import gc
import hashlib
import heapq
import itertools
import json
import os
import shutil
import time
from pathlib import Path

import duckdb
import pyarrow.parquet as pq


ASSEMBLY_VERSION = "streaming-v1"


def ao_log(message):
    print(time.strftime("%Y-%m-%d %H:%M:%S"), message, flush=True)


def ao_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


def ao_file_list(paths):
    return "[" + ",".join(ao_quote(p) for p in paths) + "]"


def ao_csv(path):
    return f"read_csv({ao_quote(path)}, delim='\\t', header=true, all_varchar=true)"


def ao_resources(work):
    free_gb = shutil.disk_usage(work).free / 1024**3
    rss = "unavailable"
    try:
        import psutil
        rss = f"{psutil.Process().memory_info().rss / 1024**3:.2f} GiB"
    except ImportError:
        pass
    ao_log(f"Resources: process RAM={rss}; disk free={free_gb:.2f} GiB")


def ao_connection(work, memory_mb=768):
    # Only external sorts and ordinary joins/counts run here. No list/string
    # aggregate, dataframe materialization, or Python-wide ID set is used.
    tmp = work / "assembly_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(work).free
    if free < 512 * 1024**2:
        raise RuntimeError("Less than 512 MiB disk free. Preserve the original Parquets and free scratch space before recovery.")
    db = duckdb.connect()
    try:
        db.execute(f"SET memory_limit = '{int(memory_mb)}MB'")
        db.execute("SET threads = 1")
        db.execute("SET preserve_insertion_order = true")
        db.execute(f"SET temp_directory = {ao_quote(tmp)}")
        spill_bytes = min(4 * 1024**3, max(256 * 1024**2, free // 2))
        db.execute(f"SET max_temp_directory_size = '{spill_bytes}B'")
        return db
    except Exception:
        db.close()
        raise


def ao_atomic_json(path, value):
    partial = path.with_name(path.name + ".partial")
    partial.write_text(json.dumps(value, indent=2), encoding="utf-8")
    os.replace(partial, path)


def ao_fingerprint(path):
    path = Path(path).resolve()
    stat = path.stat()
    return {"path": str(path), "bytes": stat.st_size, "mtime_ns": stat.st_mtime_ns}


def ao_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def ao_parquet_info(path, required):
    if not Path(path).is_file():
        raise RuntimeError(f"Required saved file is missing: {path}. Recovery cannot recreate missing predictions.")
    try:
        with pq.ParquetFile(path) as reader:
            missing = set(required) - set(reader.schema_arrow.names)
            if missing:
                raise ValueError(f"Missing columns: {sorted(missing)}")
            return reader.metadata.num_rows
    except Exception as error:
        raise RuntimeError(f"Saved Parquet is incomplete or invalid: {path}. Preserve the other files; this partition needs regeneration.") from error


def ao_sorted_file(path, work, pair_file=True):
    path = Path(path)
    identity = {"assembly_version": ASSEMBLY_VERSION, "pair_file": pair_file,
                "input": ao_fingerprint(path)}
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    cache = work / "assembly_sorted"
    cache.mkdir(exist_ok=True)
    out = cache / f"{digest}.parquet"
    marker = cache / f"{digest}.json"
    columns = ["qid", "tid"] if pair_file else ["qid"]
    expected_rows = ao_parquet_info(path, columns) if pair_file else None
    if out.exists() and marker.exists():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        actual_rows = ao_parquet_info(out, columns)
        if saved.get("identity") == identity and actual_rows == saved.get("rows"):
            if expected_rows is None or actual_rows == expected_rows:
                ao_log(f"Reuse sorted IDs: {path.name} ({actual_rows:,} rows)")
                return out, actual_rows
    ao_resources(work)
    ao_log(f"External sort with one CPU thread and 768 MB DuckDB budget: {path.name}")
    partial = out.with_name(out.stem + ".partial.parquet")
    db = ao_connection(work)
    try:
        if pair_file:
            query = f"SELECT qid, tid FROM read_parquet({ao_quote(path)}) ORDER BY qid, tid"
        else:
            expected_rows = db.execute(f"SELECT count(*) FROM {ao_csv(path)}").fetchone()[0]
            query = f"SELECT entity_id AS qid FROM {ao_csv(path)} ORDER BY entity_id"
        db.execute(f"COPY ({query}) TO {ao_quote(partial)} (FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 65536)")
    finally:
        db.close()
        gc.collect()
    actual_rows = ao_parquet_info(partial, columns)
    if actual_rows != expected_rows:
        raise RuntimeError(f"Sorted row count changed for {path}: {expected_rows} -> {actual_rows}")
    os.replace(partial, out)
    ao_atomic_json(marker, {"identity": identity, "rows": actual_rows})
    return out, actual_rows


def ao_rows(path, columns):
    previous = None
    with pq.ParquetFile(path) as reader:
        for batch in reader.iter_batches(batch_size=8192, columns=columns, use_threads=False):
            values = [batch.column(i).to_pylist() for i in range(len(columns))]
            for row in zip(*values):
                if any(value is None or not isinstance(value, str) for value in row):
                    raise RuntimeError(f"Null or non-string identifier in {path}")
                if previous is not None and row < previous:
                    raise RuntimeError(f"Sorted cache is out of order: {path}")
                previous = row
                yield row


def ao_grouped_pairs(sorted_paths, maximum):
    streams = [ao_rows(path, ["qid", "tid"]) for path in sorted_paths]
    try:
        merged = heapq.merge(*streams)
        for qid, rows in itertools.groupby(merged, key=lambda row: row[0]):
            ids = []
            previous = None
            for _, tid in rows:
                if tid == previous:
                    raise RuntimeError(f"Duplicate pair encountered for {qid}; no pairs were silently removed.")
                if not tid.startswith(("S2-", "S3-")) or any(c in tid for c in ",\t\r\n"):
                    raise RuntimeError(f"Invalid target ID for {qid}: {tid!r}")
                ids.append(tid)
                if len(ids) > maximum:
                    raise RuntimeError(f"{qid} exceeds the configured {maximum}-candidate bound. Check the selected run files.")
                previous = tid
            yield qid, ids
    finally:
        for stream in streams:
            stream.close()


def ao_check_target_ids(candidate_parts, test_dir, work):
    for source, prefix in (("source2", "S2-"), ("source3", "S3-")):
        parts = [p for p in candidate_parts if p.name.startswith(source + "_")]
        if not parts:
            raise RuntimeError(f"No {source} candidate partitions were selected")
        target_path = test_dir / f"test_{source}.tsv"
        if not target_path.is_file():
            raise RuntimeError(f"Missing official target source: {target_path}")
        ao_log(f"Check candidate IDs against the official test {source} file")
        db = ao_connection(work)
        try:
            bad_prefix = db.execute(f"SELECT count(*) FROM read_parquet({ao_file_list(parts)}) WHERE tid IS NULL OR NOT starts_with(tid, {ao_quote(prefix)})").fetchone()[0]
            invalid = db.execute(f"""
              SELECT count(*) FROM read_parquet({ao_file_list(parts)}) p
              ANTI JOIN {ao_csv(target_path)} t ON p.tid = t.entity_id
            """).fetchone()[0]
            if bad_prefix or invalid:
                raise RuntimeError(f"{source}: wrong-prefix rows={bad_prefix}, unknown-target rows={invalid}")
        finally:
            db.close()
            gc.collect()


def assemble_outputs(candidate_parts, accepted_parts, data_root, work_root,
                     max_candidates_per_entity=80, expected_accepted_counts=None):
    """Sort narrow pair files sequentially; merge/write one Source 1 at a time."""
    data_root, work = Path(data_root), Path(work_root)
    candidate_parts = [Path(p) for p in candidate_parts]
    accepted_parts = [Path(p) for p in accepted_parts]
    if not candidate_parts or len(candidate_parts) != len(accepted_parts):
        raise RuntimeError("Need a complete candidate/accepted partition pair for every source and country")
    if len({p.name for p in candidate_parts}) != len(candidate_parts):
        raise RuntimeError("Duplicate candidate partition names")
    if sorted(p.name for p in candidate_parts) != sorted(p.name for p in accepted_parts):
        raise RuntimeError("Candidate and accepted partition filenames do not describe the same run")
    candidate_count = sum(ao_parquet_info(p, ["qid", "tid"]) for p in candidate_parts)
    accepted_count = sum(ao_parquet_info(p, ["qid", "tid"]) for p in accepted_parts)
    expected_accepted_counts = expected_accepted_counts or {}
    candidate_by_name = {p.name: p for p in candidate_parts}
    completion_evidence = {}
    sidecar_flags = [p.with_name(p.name + ".complete.json").exists() for p in accepted_parts]
    if any(sidecar_flags) and not all(sidecar_flags):
        raise RuntimeError("Mixed old and new inference checkpoints. Keep separate runs together; do not assemble a mixture.")
    saved_thresholds = set()
    source_models = {}
    for part in accepted_parts:
        count = ao_parquet_info(part, ["qid", "tid"])
        sidecar = part.with_name(part.name + ".complete.json")
        candidate = candidate_by_name[part.name]
        if sidecar.exists():
            evidence = json.loads(sidecar.read_text(encoding="utf-8"))
            expected_rows = ao_parquet_info(candidate, ["qid", "tid"])
            # Content identity survives restoring the run under a different path.
            if (evidence.get("candidate_sha256") != ao_sha256(candidate)
                    or evidence.get("accepted_rows") != count
                    or evidence.get("scored_rows") != expected_rows):
                raise RuntimeError(f"Inference completion record does not match {part.name}")
            source = part.name.split("_", 1)[0]
            if evidence.get("model_file") != source + ".json":
                raise RuntimeError(f"Wrong source model in checkpoint: {part.name}")
            model_path = work / "models" / evidence["model_file"]
            if not model_path.is_file() or ao_sha256(model_path) != evidence["model_sha256"]:
                raise RuntimeError(f"Saved model changed after inference for {part.name}")
            if ao_sha256(part) != evidence.get("accepted_sha256"):
                raise RuntimeError(f"Accepted file changed after inference for {part.name}")
            saved_threshold = evidence.get("threshold")
            rounds = evidence.get("prediction_rounds")
            if not isinstance(saved_threshold, (int, float)) or not 0 <= saved_threshold <= 1:
                raise RuntimeError(f"Invalid threshold in checkpoint: {part.name}")
            if not isinstance(rounds, int) or rounds < 1:
                raise RuntimeError(f"Invalid prediction rounds in checkpoint: {part.name}")
            identity = (evidence["model_sha256"], rounds)
            if source_models.setdefault(source, identity) != identity:
                raise RuntimeError(f"Different models or prediction rounds for {source}")
            saved_thresholds.add(saved_threshold)
            completion_evidence[part.name] = evidence
        elif part.name in expected_accepted_counts:
            expected = expected_accepted_counts[part.name]
            if count != expected:
                raise RuntimeError(f"{part.name}: {count:,} rows, but completed inference log reported {expected:,}; do not use this partial file")
            completion_evidence[part.name] = {
                "accepted_rows": count, "evidence": "matched AJ's supplied completed-inference log",
                "model_and_threshold_hashes": "not available for this legacy checkpoint",
            }
        else:
            raise RuntimeError(f"No completion record for {part.name}. Supply its accepted count from a completed inference log; a valid footer alone is insufficient.")
    if len(saved_thresholds) > 1:
        raise RuntimeError("Inference checkpoints use different thresholds; do not combine runs")
    ao_log(f"ASSEMBLY START: {candidate_count:,} candidate pairs; {accepted_count:,} accepted pairs")
    ao_resources(work)
    source_files = [data_root / "test" / f"test_source{i}.tsv" for i in (1, 2, 3)]
    input_files = candidate_parts + accepted_parts + source_files
    snapshots = [ao_fingerprint(p) for p in input_files]
    # An interrupted rerun must not leave a stale success marker for new files.
    (work / "output_completion.json").unlink(missing_ok=True)
    ao_check_target_ids(candidate_parts, data_root / "test", work)
    source1 = data_root / "test" / "test_source1.tsv"
    sorted_s1, expected_s1 = ao_sorted_file(source1, work, pair_file=False)
    sorted_candidates = [ao_sorted_file(p, work)[0] for p in candidate_parts]
    sorted_accepted = [ao_sorted_file(p, work)[0] for p in accepted_parts]

    output = work / "output"
    output.mkdir(exist_ok=True)
    candidate_final = output / "candidate_pairs.tsv"
    matching_final = output / "matching_results.tsv"
    candidate_partial = output / "candidate_pairs.tsv.partial"
    matching_partial = output / "matching_results.tsv.partial"
    chash, mhash = hashlib.sha256(), hashlib.sha256()
    cgroups = ao_grouped_pairs(sorted_candidates, max_candidates_per_entity)
    mgroups = ao_grouped_pairs(sorted_accepted, max_candidates_per_entity)
    source_rows = ao_rows(sorted_s1, ["qid"])
    cgroup, mgroup = next(cgroups, None), next(mgroups, None)
    rows = seen_candidates = seen_matches = empty_candidates = 0
    previous_qid = None
    ao_log("Stream both TSV files; validate coverage, duplicates and match/candidate subsets")
    try:
        with candidate_partial.open("wb", buffering=1024**2) as cf, matching_partial.open("wb", buffering=1024**2) as mf:
            cheader = b"source1_entity_id\tcandidate_entity_ids\n"
            mheader = b"source1_entity_id\tmatched_entity_ids\n"
            cf.write(cheader); chash.update(cheader)
            mf.write(mheader); mhash.update(mheader)
            for (qid,) in source_rows:
                if qid == previous_qid or not qid.startswith("S1-") or any(c in qid for c in ",\t\r\n"):
                    raise RuntimeError("Invalid or duplicated Source 1 ID in official input")
                if cgroup is not None and cgroup[0] < qid:
                    raise RuntimeError(f"Candidate Source 1 ID absent from test: {cgroup[0]}")
                if mgroup is not None and mgroup[0] < qid:
                    raise RuntimeError(f"Accepted Source 1 ID absent from test: {mgroup[0]}")
                cids, mids = [], []
                if cgroup is not None and cgroup[0] == qid:
                    cids = cgroup[1]
                    cgroup = next(cgroups, None)
                if mgroup is not None and mgroup[0] == qid:
                    mids = mgroup[1]
                    mgroup = next(mgroups, None)
                if not set(mids).issubset(cids):
                    raise RuntimeError(f"Accepted ID missing from the scored candidate set for {qid}")
                cline = (qid + "\t" + ",".join(cids) + "\n").encode("utf-8")
                mline = (qid + "\t" + ",".join(mids) + "\n").encode("utf-8")
                cf.write(cline); chash.update(cline)
                mf.write(mline); mhash.update(mline)
                rows += 1
                seen_candidates += len(cids)
                seen_matches += len(mids)
                empty_candidates += not cids
                previous_qid = qid
                if rows % 100_000 == 0:
                    ao_log(f"  assembled {rows:,}/{expected_s1:,} Source 1 entities")
            if cgroup is not None or mgroup is not None:
                raise RuntimeError("Pairs remain for Source 1 IDs absent from the official test source")
            if (rows, seen_candidates, seen_matches) != (expected_s1, candidate_count, accepted_count):
                raise RuntimeError("Row totals differ from the complete input Parquet/source counts")
            cf.flush(); os.fsync(cf.fileno())
            mf.flush(); os.fsync(mf.fileno())
    finally:
        source_rows.close(); cgroups.close(); mgroups.close()
    if snapshots != [ao_fingerprint(p) for p in input_files]:
        raise RuntimeError("Input files changed during assembly; no completion marker was written")
    os.replace(candidate_partial, candidate_final)
    os.replace(matching_partial, matching_final)
    result = {
        "assembly_version": ASSEMBLY_VERSION, "source1_rows": rows,
        "candidate_pairs": seen_candidates, "accepted_pairs": seen_matches,
        "source1_without_candidates": empty_candidates,
        "validation": "PASS: exact Source 1 coverage, target existence, unique pairs and match subsets",
        "official_validator_run": False,
        "inference_completion_evidence": completion_evidence,
        "input_files": snapshots,
        "outputs": {
            candidate_final.name: {"bytes": candidate_final.stat().st_size, "sha256": chash.hexdigest()},
            matching_final.name: {"bytes": matching_final.stat().st_size, "sha256": mhash.hexdigest()},
        },
    }
    ao_atomic_json(work / "output_completion.json", result)
    ao_log(f"OUTPUT COMPLETE: {rows:,} Source 1 rows, {seen_matches:,} final links. Files: {output}")
    ao_resources(work)
    return result


def recover_saved_outputs(data_root=None, work_root=None,
                          candidate_tag="v3_64_512_3_40", expected_accepted_counts=None):
    work = Path(work_root or "/kaggle/working/aj_entity_resolution")
    if not work.is_dir():
        raise RuntimeError(f"Saved run directory is missing: {work}. Restore its Parquets before recovery.")
    if data_root is None:
        roots = sorted({p.parent.parent.resolve() for p in Path("/kaggle/input").glob("**/dataset/test/test_source1.tsv")})
        if len(roots) != 1:
            raise RuntimeError(f"Expected one attached official dataset; found {roots}. Set data_root explicitly.")
        data_root = roots[0]
    data_root = Path(data_root)
    db = ao_connection(work)
    try:
        countries = [row[0] for row in db.execute(f"SELECT DISTINCT country FROM {ao_csv(data_root / 'test' / 'test_source1.tsv')} ORDER BY country").fetchall()]
    finally:
        db.close()
    candidates, accepted = [], []
    for country in countries:
        country_key = hashlib.sha1(country.encode()).hexdigest()[:10]
        for source in ("source2", "source3"):
            filename = f"{source}_{country_key}_{candidate_tag}.parquet"
            candidates.append(work / "test_candidates" / filename)
            accepted.append(work / "accepted" / filename)
    ao_log("RECOVERY ONLY: reuse existing Parquets; no training, inference or GPU imports")
    return assemble_outputs(candidates, accepted, data_root, work,
                            expected_accepted_counts=expected_accepted_counts)

## 1. Locate the official files and freeze this run

The candidate engine is copied from team branch `feat/data-candidates` at commit `09bb350a7dd52393cf8d27e002c8ffdf67840685` (Apache-2.0). This notebook uses K64/A32 retrieval, a 256-row per-source staged shortlist, and a 128-row combined final set; its TSV reader accepts the gzip country shards written below. The complete upstream Apache-2.0 license is included in the final notebook cell.

In [ ]:
import csv
import gc
import gzip
import hashlib
import json
import os
import random
import shutil
import sys
import time
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(ENGINE_DIR))
import candidates as retrieval
import data as source_data
from output_assembly import (
    ao_atomic_json, ao_connection, ao_file_list, ao_fingerprint, ao_log,
    ao_parquet_info, ao_resources, ao_sha256, assemble_outputs,
)

input_roots = sorted({
    path.parent.parent.resolve()
    for path in Path("/kaggle/input").glob("**/dataset/train/train_source1.tsv")
})
if len(input_roots) != 1:
    raise RuntimeError(f"Attach exactly one official dataset; found {input_roots}")
DATA_ROOT = input_roots[0]
REQUIRED = [
    DATA_ROOT / split / f"{split}_source{source}.tsv"
    for split in ("train", "test") for source in (1, 2, 3)
] + [DATA_ROOT / "train" / "train_ground_truth.tsv"]
missing = [str(path) for path in REQUIRED if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Official challenge files are missing: {missing}")

def file_id(path):
    stat = Path(path).stat()
    return {"name": Path(path).name, "bytes": stat.st_size, "mtime_ns": stat.st_mtime_ns}

# Content hashes make a saved checkpoint specific to the exact official files,
# and relative keys keep its identity stable if Kaggle changes the mount name.
INPUT_IDS = {
    path.relative_to(DATA_ROOT).as_posix(): {
        "bytes": path.stat().st_size, "sha256": ao_sha256(path),
    }
    for path in REQUIRED
}

# Broad-token routes from the completed 10k full-target experiment, with
# a wider staged shortlist and final cap to raise the retrieval ceiling.
SETTINGS = retrieval.Settings(
    name_k=64, address_k=32, cap=128, stage_cap=256,
    indexed_grams=16, query_grams=32, gram_selection="rarest",
    pair_max_df=5_000, pair_max_hits=2_000,
    single_max_df=512, single_tokens=2, workers=2,
)
TARGET_MACRO_F05 = 0.989
FORCE_TEST_RUN_BELOW_TARGET = False
QUERY_POLICY = {
    "stable_hash": "sha256(qid) first 8 hex digits modulo 100",
    "valid_buckets": "0-4", "holdout_buckets": "5-9",
    "fit_buckets": "10-49", "previously_inspected_sample_excluded": True,
}
FIT_NEGATIVES_PER_QUERY = 16
TRAINING_POLICY = {
    "fit_negatives_per_query": FIT_NEGATIVES_PER_QUERY,
    "fit_negative_selection": "highest max(name/address Jaro-Winkler and cosine), tid tiebreak",
    "fit_query_weight": "each query's retained pair weights sum to one",
    "validation_and_holdout": "all retrieved pairs, no negative sampling",
}
FEATURE_PIPELINE_VERSION = "duckdb-pair-features-v1"
XGBOOST_PARAMS = {
    "objective": "binary:logistic", "eval_metric": "aucpr",
    "tree_method": "hist", "max_depth": 8, "eta": 0.07,
    "max_bin": 256, "subsample": 0.85, "colsample_bytree": 0.9,
    "min_child_weight": 8, "lambda": 4.0,
}
XGBOOST_TRAINING = {"num_boost_round": 1200, "early_stopping_rounds": 75}
LIGHTGBM_PARAMS = {
    "objective": "binary", "n_estimators": 1500, "learning_rate": 0.05,
    "num_leaves": 63, "subsample": 0.85, "subsample_freq": 1,
    "colsample_bytree": 0.9, "reg_lambda": 4.0,
    "min_child_samples": 50, "verbosity": -1,
}
LIGHTGBM_TRAINING = {"early_stopping_rounds": 75, "eval_metric": "binary_logloss"}
MODEL_CONFIGURATION = {
    "xgboost": XGBOOST_PARAMS, "xgboost_training": XGBOOST_TRAINING,
    "lightgbm": LIGHTGBM_PARAMS, "lightgbm_training": LIGHTGBM_TRAINING,
    "source_seed_rule": "2026 + source number",
}
RUNTIME_VERSIONS = {
    "duckdb": duckdb.__version__, "xgboost": xgb.__version__,
    "lightgbm": getattr(lgb, "__version__", None),
    "numpy": np.__version__, "pandas": pd.__version__, "pyarrow": pa.__version__,
    "torch": torch.__version__,
}
BUILD_CODE_SHA256 = "6524b8fc7a63dd5479da0621242ccc4a3464e84faf6b0e59c3e52f396dffc712"
PIPELINE_VERSION = "kaggle-master-style-v3"
run_identity = {
    "version": PIPELINE_VERSION, "build_code_sha256": BUILD_CODE_SHA256,
    "feature_pipeline_version": FEATURE_PIPELINE_VERSION,
    "model_configuration": MODEL_CONFIGURATION, "runtime_versions": RUNTIME_VERSIONS,
    "source_commit": "09bb350a7dd52393cf8d27e002c8ffdf67840685",
    "settings": vars(SETTINGS), "query_policy": QUERY_POLICY,
    "training_policy": TRAINING_POLICY, "inputs": INPUT_IDS,
}
RUN_ID = hashlib.sha256(json.dumps(run_identity, sort_keys=True).encode()).hexdigest()[:12]
WORK = Path("/kaggle/working") / f"amazon_ml_master_{RUN_ID}"
checkpoint_inputs = [
    path for path in Path("/kaggle/input").glob(f"**/amazon_ml_master_{RUN_ID}")
    if (path / "run_identity.json").is_file()
]
saved_identity = WORK / "run_identity.json"
if saved_identity.is_file():
    if json.loads(saved_identity.read_text(encoding="utf-8")) != run_identity:
        raise RuntimeError(f"Existing work directory has a different run identity: {WORK}")
elif checkpoint_inputs:
    if len(checkpoint_inputs) != 1:
        raise RuntimeError(f"Found {len(checkpoint_inputs)} matching attached checkpoints; attach only one")
    checkpoint = checkpoint_inputs[0]
    checkpoint_identity = json.loads((checkpoint / "run_identity.json").read_text(encoding="utf-8"))
    if checkpoint_identity != run_identity:
        raise RuntimeError("Attached checkpoint content identity does not match the official input files")
    def ignore_checkpoint_cache(directory, names):
        return {name for name in names
                if name == "tmp" or ".partial" in name.lower() or name.lower().endswith(".tmp")}
    checkpoint_bytes = sum(
        item.stat().st_size for item in checkpoint.rglob("*")
        if item.is_file() and ".partial" not in item.name.lower()
        and not item.name.lower().endswith(".tmp")
        and "tmp" not in item.relative_to(checkpoint).parts
    )
    free_bytes = shutil.disk_usage(Path("/kaggle/working")).free
    if checkpoint_bytes > free_bytes * 0.85:
        raise RuntimeError(
            f"Matching checkpoint needs about {checkpoint_bytes / 1024**3:.1f} GiB, "
            f"but only {free_bytes / 1024**3:.1f} GiB is free under /kaggle/working. "
            "Free disk or attach a smaller completed checkpoint before resuming."
        )
    shutil.copytree(checkpoint, WORK, dirs_exist_ok=True, ignore=ignore_checkpoint_cache)
    print(f"Restored matching checkpoint from {checkpoint}", flush=True)
for folder in ("target_shards", "candidate_files", "retrieval", "train_features",
               "test_features", "models", "accepted", "tmp", "output"):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
(WORK / "run_identity.json").write_text(json.dumps(run_identity, indent=2), encoding="utf-8")
OUTPUT = WORK / "output"
THREADS = max(2, min(8, os.cpu_count() or 4))

print(f"Official files found. Run ID: {RUN_ID}; work directory: {WORK}")
print(f"Free disk now: {shutil.disk_usage(WORK).free / 1024**3:.1f} GiB")

## 2. Make compact country target shards

Country equality is supported by the observed training labels, which showed no cross-country true links. We split each target source into gzip shards in one streaming pass, so each query is ranked only against same-country records and the candidate cap is not consumed by unrelated countries. France has no labeled training examples, so cross-country blocking remains an unverified assumption for France. The complete source files remain the authoritative inputs for feature joins and final validation.

In [ ]:
SOURCE_COLUMNS = list(source_data.SOURCE_COLUMNS)

def country_key(country):
    return hashlib.sha1(country.encode("utf-8")).hexdigest()[:12]

def target_shard_path(split, source, country):
    return WORK / "target_shards" / split / f"source{source}_{country_key(country)}.tsv.gz"

def feature_files_ready(split):
    suffix = "fit" if split == "train" else "all"
    for source in (2, 3):
        path = WORK / f"{split}_features" / f"source{source}_{suffix}.parquet"
        marker = path.with_suffix(".complete.json")
        if not path.is_file() or not marker.is_file():
            return False
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if (saved.get("identity", {}).get("run_id") != RUN_ID
                or saved.get("sha256") != ao_sha256(path)):
            return False
    return True

def partition_target_file(split, source):
    raw_path = DATA_ROOT / split / f"{split}_source{source}.tsv"
    out_dir = WORK / "target_shards" / split
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir / f"source{source}.complete.json"
    identity = {"input": file_id(raw_path), "source": source, "split": split,
                "columns": SOURCE_COLUMNS, "gzip": True}
    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8")).get("identity") == identity:
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if all(Path(item).is_file() for item in saved.get("shards", [])):
            return saved["countries"]

    handles, writers, counts = {}, {}, defaultdict(int)
    temp_paths = {}
    with raw_path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.reader(stream, delimiter="\t", strict=True)
        header = next(reader, None)
        if header != SOURCE_COLUMNS:
            raise ValueError(f"Unexpected columns in {raw_path}: {header}")
        for line_number, row in enumerate(reader, start=2):
            if len(row) != 4 or not row[3].strip():
                raise ValueError(f"Bad row or empty country at {raw_path}:{line_number}")
            if not row[0].startswith(f"S{source}-"):
                raise ValueError(f"Wrong source ID at {raw_path}:{line_number}: {row[0]}")
            country = row[3]
            key = country_key(country)
            if country not in writers:
                final_path = out_dir / f"source{source}_{key}.tsv.gz"
                temp_path = final_path.with_name(final_path.name + ".partial")
                handle = gzip.open(temp_path, "wt", encoding="utf-8", newline="", compresslevel=3)
                writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
                writer.writerow(SOURCE_COLUMNS)
                handles[country] = handle
                writers[country] = writer
                temp_paths[country] = (temp_path, final_path)
            writers[country].writerow(row)
            counts[country] += 1
    for handle in handles.values():
        handle.close()
    for temp_path, final_path in temp_paths.values():
        os.replace(temp_path, final_path)
    saved = {
        "identity": identity, "countries": sorted(counts),
        "row_counts": dict(counts),
        "shards": [str(target_shard_path(split, source, country)) for country in sorted(counts)],
    }
    ao_atomic_json(marker, saved)
    print(f"Target shards {split} S{source}: {sum(counts.values()):,} rows across {len(counts)} countries")
    return saved["countries"]

target_countries = {}
if feature_files_ready("train"):
    print("Reuse verified train feature files; skip train target-shard rebuild", flush=True)
else:
    for source in (2, 3):
        target_countries[("train", source)] = set(partition_target_file("train", source))

## 3. Generate checkpointed candidates

The training queries are selected deterministically. The 10k retrieval-development sample and the next 50k inspected queries from the earlier experiment are excluded entirely. Each country/source writes completed compressed candidates only after its retrieval chunks finish; rerunning this cell reuses completed work and resumes missing chunks. The final cap is 128 combined across Sources 2 and 3, with up to 256 ranked proposals staged per source first.

In [ ]:
def sha_bucket(entity_id):
    return int(hashlib.sha256(entity_id.encode("utf-8")).hexdigest()[:8], 16) % 100

def load_train_query_cohorts():
    records = retrieval._load_sorted_source1(source_data.source_path(DATA_ROOT, "train", 1), None)
    sampled = random.Random(20260926).sample(range(len(records)), min(60_000, len(records)))
    inspected_ids = {records[index].entity_id for index in sampled}
    grouped, scope_rows = defaultdict(list), []
    for record in records:
        if record.entity_id in inspected_ids:
            continue
        bucket = sha_bucket(record.entity_id)
        cohort = "valid" if bucket < 5 else "holdout" if bucket < 10 else "fit" if bucket < 50 else None
        if cohort:
            grouped[record.country].append(record)
            scope_rows.append((record.entity_id, cohort))
    scope_rows.sort()
    scope_path = WORK / "train_query_scope.tsv"
    with scope_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream, delimiter="\t", lineterminator="\n")
        writer.writerow(["qid", "cohort"])
        writer.writerows(scope_rows)
    del records
    gc.collect()
    print({"selected_train_queries": len(scope_rows),
           "fit": sum(cohort == "fit" for _, cohort in scope_rows),
           "valid": sum(cohort == "valid" for _, cohort in scope_rows),
           "holdout": sum(cohort == "holdout" for _, cohort in scope_rows),
           "previously_inspected_excluded": len(inspected_ids)})
    return dict(grouped), scope_path, len(scope_rows)

def load_test_queries():
    records = retrieval._load_sorted_source1(source_data.source_path(DATA_ROOT, "test", 1), None)
    grouped = defaultdict(list)
    for record in records:
        grouped[record.country].append(record)
    print(f"Test Source 1 queries: {len(records):,} across {len(grouped)} countries")
    del records
    gc.collect()
    return dict(grouped)

TRAIN_QUERIES, SCOPE_PATH, TRAIN_QUERY_COUNT = load_train_query_cohorts()

def query_digest(records):
    digest = hashlib.sha256()
    for record in records:
        digest.update(record.entity_id.encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()

def write_country_candidates(split, country, records):
    key = country_key(country)
    output_dir = WORK / "candidate_files" / split
    output_dir.mkdir(parents=True, exist_ok=True)
    out2 = output_dir / f"source2_{key}.tsv.gz"
    out3 = output_dir / f"source3_{key}.tsv.gz"
    marker = output_dir / f"country_{key}.complete.json"
    country_work = WORK / "retrieval" / split / key
    country_work.mkdir(parents=True, exist_ok=True)
    target_paths = {source: target_shard_path(split, source, country) for source in (2, 3)}
    identity = {
        "run_id": RUN_ID, "split": split, "country": country,
        "query_count": len(records), "query_sha256": query_digest(records),
        "settings": vars(SETTINGS),
        "targets": {str(source): file_id(path) if path.is_file() else None
                    for source, path in target_paths.items()},
    }
    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8")).get("identity") == identity:
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if out2.is_file() and out3.is_file() and ao_sha256(out2) == saved.get("source2_sha256") and ao_sha256(out3) == saved.get("source3_sha256"):
            return saved

    for source in (2, 3):
        path = target_paths[source]
        if path.is_file() and path.stat().st_size > 40:
            retrieval._retrieve_source(records, path, source, SETTINGS, country_work, None)

    temp2 = out2.with_name(out2.name + ".partial")
    temp3 = out3.with_name(out3.name + ".partial")
    counts = {2: 0, 3: 0}
    with gzip.open(temp2, "wt", encoding="utf-8", newline="", compresslevel=3) as h2, \
         gzip.open(temp3, "wt", encoding="utf-8", newline="", compresslevel=3) as h3:
        writer2, writer3 = csv.writer(h2, delimiter="\t", lineterminator="\n"), csv.writer(h3, delimiter="\t", lineterminator="\n")
        writer2.writerow(retrieval.HEADER)
        writer3.writerow(retrieval.HEADER)
        chunks = (len(records) + SETTINGS.query_chunk - 1) // SETTINGS.query_chunk
        for chunk in range(chunks):
            p2 = retrieval._part_path(country_work, 2, chunk)
            p3 = retrieval._part_path(country_work, 3, chunk)
            groups2 = retrieval._read_part(p2) if p2.is_file() else {}
            groups3 = retrieval._read_part(p3) if p3.is_file() else {}
            batch = records[chunk * SETTINGS.query_chunk:(chunk + 1) * SETTINGS.query_chunk]
            for record in batch:
                selected = retrieval._choose_rows(
                    groups2.get(record.entity_id, []), groups3.get(record.entity_id, []), SETTINGS.cap
                )
                for row in selected:
                    if row[1].startswith("S2-"):
                        writer2.writerow(row)
                        counts[2] += 1
                    elif row[1].startswith("S3-"):
                        writer3.writerow(row)
                        counts[3] += 1
                    else:
                        raise RuntimeError(f"Unexpected target ID: {row[1]}")
            del groups2, groups3
    os.replace(temp2, out2)
    os.replace(temp3, out3)
    saved = {
        "identity": identity, "source1_queries": len(records),
        "source2_pairs": counts[2], "source3_pairs": counts[3],
        "source2_sha256": ao_sha256(out2), "source3_sha256": ao_sha256(out3),
    }
    ao_atomic_json(marker, saved)
    print(f"Candidates {split} {country}: S2={counts[2]:,}; S3={counts[3]:,}; cap={SETTINGS.cap}", flush=True)
    return saved

def run_candidate_split(split, grouped):
    feature_dir = WORK / f"{split}_features"
    feature_suffix = "fit" if split == "train" else "all"
    feature_ready = True
    for source in (2, 3):
        feature_path = feature_dir / f"source{source}_{feature_suffix}.parquet"
        feature_marker = feature_path.with_suffix(".complete.json")
        if not feature_path.is_file() or not feature_marker.is_file():
            feature_ready = False
            break
        saved = json.loads(feature_marker.read_text(encoding="utf-8"))
        if (saved.get("identity", {}).get("run_id") != RUN_ID
                or saved.get("sha256") != ao_sha256(feature_path)):
            feature_ready = False
            break
    if feature_ready:
        print(f"Reuse completed {split} feature files; raw country candidate files are no longer needed.")
        return
    started = time.monotonic()
    countries = sorted(grouped)
    total_queries = sum(len(grouped[country]) for country in countries)
    processed_queries = 0
    for i, country in enumerate(countries, start=1):
        write_country_candidates(split, country, grouped[country])
        processed_queries += len(grouped[country])
        if i == 1 or i % 5 == 0 or i == len(countries):
            elapsed = max(1.0, time.monotonic() - started)
            rate = processed_queries / elapsed
            eta_hours = (total_queries - processed_queries) / max(rate, 1e-9) / 3600
            print(f"{split}: countries {i}/{len(countries)}; queries {processed_queries:,}/{total_queries:,}; "
                  f"elapsed {elapsed/3600:.2f}h; rough remaining {eta_hours:.2f}h", flush=True)
    candidate_files = sorted((WORK / "candidate_files" / split).glob("source[23]_*.tsv.gz"))
    print(f"{split}: {len(candidate_files)} compressed candidate files; free disk {shutil.disk_usage(WORK).free / 1024**3:.1f} GiB")

## 4. Recompute pair features on the exact candidate set

Candidate generation and ranking are separated. The features below are calculated from the original official source files after joining only the selected candidate IDs. This avoids training on one candidate distribution and submitting another. The XGBoost/LightGBM models are trained from scratch on this broader candidate set.

Only train retrieval and features are built in this section. Test retrieval and features are deferred until validation and the untouched holdout establish that the measured retrieval upper bound can plausibly reach the 98.9% target.

In [ ]:
import duckdb

CHANNELS = [
    "exact_name", "exact_address", "name_signature", "compact_name",
    "rare_name", "rare_address", "name_pair", "address_pair", "cross_token",
    "single_name", "single_address", "name_char", "address_char",
]
FEATURES = [
    "name_jw", "address_jw", "name_jaccard", "address_jaccard",
    "name_length_ratio", "address_length_ratio", "number_equal",
    "number_both_present", "exact_name", "exact_address",
    "name_cosine", "address_cosine", "channel_count",
] + [f"channel_{channel}" for channel in CHANNELS]

def sql_quote(value):
    return "'" + str(value).replace("'", "''") + "'"

def csv_relation(path):
    return f"read_csv({sql_quote(path)}, delim='\\t', header=true, all_varchar=true)"

def candidate_relation(paths):
    if not paths:
        raise RuntimeError("No candidate files found for a source")
    file_list = "[" + ",".join(sql_quote(path) for path in paths) + "]"
    return f"read_csv({file_list}, delim='\\t', header=true, all_varchar=true, union_by_name=true)"

def feature_connection():
    db = duckdb.connect()
    db.execute("SET memory_limit='8GB'")
    db.execute(f"SET threads={min(8, THREADS)}")
    db.execute(f"SET temp_directory={sql_quote(WORK / 'tmp')}")
    db.execute("SET preserve_insertion_order=false")
    return db

def feature_query(split, source, candidate_files, output_path):
    query_path = DATA_ROOT / split / f"{split}_source1.tsv"
    target_path = DATA_ROOT / split / f"{split}_source{source}.tsv"
    cohort_join = ""
    cohort_source_select = ""
    cohort_select = ""
    cohort_filter = ""
    if split == "train":
        cohort_join = f"LEFT JOIN {csv_relation(SCOPE_PATH)} scope ON scope.qid=c.qid"
        cohort_source_select = ", scope.cohort AS cohort"
        cohort_select = ", cohort"
        cohort_filter = "AND scope.cohort IS NOT NULL"

    channel_select = ",\n".join(
        f"CASE WHEN strpos('|' || c.retrieval_channels || '|', '|{name}|') > 0 THEN 1 ELSE 0 END::TINYINT AS channel_{name}"
        for name in CHANNELS
    )
    query = f"""
    COPY (
      WITH c AS (
        SELECT source1_entity_id AS qid, candidate_entity_id AS tid,
               CAST(name_cosine AS FLOAT) AS name_cosine,
               CAST(address_cosine AS FLOAT) AS address_cosine,
               retrieval_channels
        FROM {candidate_relation(candidate_files)}
        WHERE starts_with(candidate_entity_id, {sql_quote('S' + str(source) + '-')})
      ), q AS (
        SELECT entity_id AS rid, country,
          trim(regexp_replace(regexp_replace(
            ' ' || trim(regexp_replace(lower(strip_accents(coalesce(business_name,''))), '[^a-z0-9]+', ' ', 'g')) || ' ',
            ' (incorporated|inc|corporation|corp|limited|ltd|llc|llp|private|pvt) ', ' ', 'g'),
            ' +', ' ', 'g')) AS nm,
          trim(regexp_replace(lower(strip_accents(coalesce(business_address,''))), '[^a-z0-9]+', ' ', 'g')) AS ad,
          regexp_extract(coalesce(business_address,''), '[0-9]+') AS num
        FROM {csv_relation(query_path)}
      ), t AS (
        SELECT entity_id AS rid, country,
          trim(regexp_replace(regexp_replace(
            ' ' || trim(regexp_replace(lower(strip_accents(coalesce(business_name,''))), '[^a-z0-9]+', ' ', 'g')) || ' ',
            ' (incorporated|inc|corporation|corp|limited|ltd|llc|llp|private|pvt) ', ' ', 'g'),
            ' +', ' ', 'g')) AS nm,
          trim(regexp_replace(lower(strip_accents(coalesce(business_address,''))), '[^a-z0-9]+', ' ', 'g')) AS ad,
          regexp_extract(coalesce(business_address,''), '[0-9]+') AS num
        FROM {csv_relation(target_path)}
      ), joined AS (
        SELECT c.qid, c.tid, c.name_cosine, c.address_cosine, c.retrieval_channels,
               q.nm AS qnm, q.ad AS qad, q.num AS qnum,
               t.nm AS tnm, t.ad AS tad, t.num AS tnum
               {cohort_source_select}
        FROM c JOIN q ON q.rid=c.qid JOIN t ON t.rid=c.tid AND t.country=q.country
        {cohort_join}
        WHERE true {cohort_filter}
      )
      SELECT qid, tid,
        jaro_winkler_similarity(qnm, tnm)::FLOAT AS name_jw,
        CASE WHEN qad='' OR tad='' THEN 0 ELSE jaro_winkler_similarity(qad,tad) END::FLOAT AS address_jw,
        CASE WHEN qnm='' OR tnm='' THEN 0 ELSE jaccard(qnm,tnm) END::FLOAT AS name_jaccard,
        CASE WHEN qad='' OR tad='' THEN 0 ELSE jaccard(qad,tad) END::FLOAT AS address_jaccard,
        (least(length(qnm),length(tnm))::FLOAT / greatest(1,length(qnm),length(tnm)))::FLOAT AS name_length_ratio,
        (least(length(qad),length(tad))::FLOAT / greatest(1,length(qad),length(tad)))::FLOAT AS address_length_ratio,
        CASE WHEN qnum<>'' AND qnum=tnum THEN 1 ELSE 0 END::FLOAT AS number_equal,
        CASE WHEN qnum<>'' AND tnum<>'' THEN 1 ELSE 0 END::FLOAT AS number_both_present,
        CASE WHEN qnm<>'' AND qnm=tnm THEN 1 ELSE 0 END::FLOAT AS exact_name,
        CASE WHEN qad<>'' AND qad=tad THEN 1 ELSE 0 END::FLOAT AS exact_address,
        name_cosine, address_cosine,
        (length(retrieval_channels)-length(replace(retrieval_channels,'|',''))
          + CASE WHEN retrieval_channels='' THEN 0 ELSE 1 END)::FLOAT AS channel_count,
        {channel_select}
        {cohort_select}
      FROM joined c
    ) TO {sql_quote(output_path)} (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 65536)
    """
    return query

FEATURE_FILES = {"train": {}, "test": {}}
FEATURE_MARKERS = {"train": {}, "test": {}}

def build_feature_split(split, query_groups=None):
    if split not in {"train", "test"}:
        raise ValueError(f"Unsupported split: {split}")
    if not feature_files_ready(split):
        for source in (2, 3):
            partition_target_file(split, source)
        if query_groups is None:
            query_groups = TRAIN_QUERIES if split == "train" else load_test_queries()
        run_candidate_split(split, query_groups)
        del query_groups
    for source in (2, 3):
        files = sorted((WORK / "candidate_files" / split).glob(f"source{source}_*.tsv.gz"))
        output_dir = WORK / f"{split}_features"
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / f"source{source}_{'fit' if split == 'train' else 'all'}.parquet"
        marker = output_path.with_suffix(".complete.json")
        prior = json.loads(marker.read_text(encoding="utf-8")) if marker.is_file() else {}
        if (output_path.is_file() and prior.get("identity", {}).get("run_id") == RUN_ID
                and prior.get("sha256") == ao_sha256(output_path)):
            ao_parquet_info(output_path, ["qid", "tid", *FEATURES])
            FEATURE_FILES[split][source] = output_path
            FEATURE_MARKERS[split][source] = marker
            for candidate_file in files:
                candidate_file.unlink(missing_ok=True)
            print(f"Reuse features: {output_path.name}; remove regenerated candidate intermediates")
            continue
        candidate_identity = [(str(p), p.stat().st_size, ao_sha256(p)) for p in files]
        identity = {"run_id": RUN_ID, "split": split, "source": source,
                    "feature_names": FEATURES, "candidates": candidate_identity}
        free_gb = shutil.disk_usage(WORK).free / 1024**3
        print(f"Build {split} S{source} features from {len(files)} country files; free disk={free_gb:.1f} GiB", flush=True)
        if output_path.exists():
            output_path.unlink()
        db = feature_connection()
        try:
            db.execute(feature_query(split, source, [str(p) for p in files], output_path))
        finally:
            db.close()
            gc.collect()
        rows = ao_parquet_info(output_path, ["qid", "tid", *FEATURES])
        ao_atomic_json(marker, {"identity": identity, "rows": rows, "sha256": ao_sha256(output_path)})
        FEATURE_FILES[split][source] = output_path
        FEATURE_MARKERS[split][source] = marker
        for candidate_file in files:
            candidate_file.unlink(missing_ok=True)
        print(f"Feature rows {split} S{source}: {rows:,}; file={output_path.stat().st_size/1024**3:.2f} GiB", flush=True)
    retrieval_root = WORK / "retrieval" / split
    if retrieval_root.exists():
        shutil.rmtree(retrieval_root)
        print(f"Removed completed {split} retrieval intermediates; verified feature Parquets retain the candidate set.")
    if feature_files_ready(split):
        shard_root = WORK / "target_shards" / split
        if shard_root.exists():
            shutil.rmtree(shard_root)
            print(f"Removed completed {split} target shards; source files and feature Parquets remain authoritative.")

build_feature_split("train", TRAIN_QUERIES)
del TRAIN_QUERIES

## 5. Fit XGBoost and LightGBM pair matchers

Each Source 1 query stays wholly inside one split. XGBoost uses each T4 in turn; LightGBM is a CPU challenger when the Kaggle image already includes it. We select the matcher and threshold using validation macro F0.5, then report that choice on the untouched holdout. No test labels are used.

In [ ]:
FEATURES_SQL = ", ".join(f"f.{name}" for name in FEATURES)
TRUTH_PAIRS = WORK / "truth_pairs.parquet"
if not TRUTH_PAIRS.is_file():
    db = feature_connection()
    try:
        db.execute(f"""
          COPY (
            SELECT DISTINCT source1_entity_id AS qid, trim(tid) AS tid
            FROM (
              SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS tid
              FROM {csv_relation(DATA_ROOT / 'train' / 'train_ground_truth.tsv')}
              WHERE matched_entity_ids IS NOT NULL AND matched_entity_ids <> ''
            )
            WHERE trim(tid) <> ''
          ) TO {sql_quote(TRUTH_PAIRS)} (FORMAT PARQUET, COMPRESSION ZSTD)
        """)
    finally:
        db.close()
ao_parquet_info(TRUTH_PAIRS, ["qid", "tid"])

MODEL_DIR = WORK / "models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_SCORE_FILES = {"valid": {}, "holdout": {}}

def labeled_rows(source, cohort):
    feature_path = FEATURE_FILES["train"][source]
    db = feature_connection()
    try:
        labeled = f"""
          SELECT f.qid, f.tid, {FEATURES_SQL},
                 CAST(l.tid IS NOT NULL AS UTINYINT) AS y
          FROM read_parquet({sql_quote(feature_path)}) f
          LEFT JOIN read_parquet({sql_quote(TRUTH_PAIRS)}) l ON f.qid=l.qid AND f.tid=l.tid
          WHERE f.cohort={sql_quote(cohort)}
        """
        if cohort != "fit":
            return db.execute(labeled).df()
        return db.execute(f"""
          WITH labeled AS ({labeled}), ranked_negatives AS (
            SELECT *, row_number() OVER (
              PARTITION BY qid
              ORDER BY greatest(coalesce(name_jw, 0), coalesce(address_jw, 0),
                                coalesce(name_cosine, 0), coalesce(address_cosine, 0)) DESC,
                       tid
            ) AS negative_rank
            FROM labeled WHERE y=0
          )
          SELECT * EXCLUDE (negative_rank) FROM ranked_negatives
          WHERE negative_rank <= {FIT_NEGATIVES_PER_QUERY}
          UNION ALL
          SELECT * FROM labeled WHERE y=1
        """).df()
    finally:
        db.close()

def model_identity(source):
    return {
        "run_id": RUN_ID, "source": source, "features": FEATURES,
        "train_features_sha256": ao_sha256(FEATURE_FILES["train"][source]),
        "truth_sha256": ao_sha256(TRUTH_PAIRS),
        "xgboost": xgb.__version__, "lightgbm": getattr(lgb, "__version__", None),
        "fit_cohort": "fit", "early_stop_cohort": "valid", "has_lightgbm": HAS_LIGHTGBM,
    }

def train_source_model(source, gpu):
    marker = MODEL_DIR / f"source{source}.training.complete.json"
    valid_path = MODEL_DIR / f"source{source}_valid_scores.parquet"
    holdout_path = MODEL_DIR / f"source{source}_holdout_scores.parquet"
    expected = model_identity(source)
    if marker.is_file() and valid_path.is_file() and holdout_path.is_file():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if saved.get("identity") == expected:
            MODEL_SCORE_FILES["valid"][source] = valid_path
            MODEL_SCORE_FILES["holdout"][source] = holdout_path
            print(f"Reuse trained source{source} models and validation scores")
            return saved

    train_df = labeled_rows(source, "fit")
    if train_df.empty or train_df.y.nunique() < 2:
        raise RuntimeError(f"Source {source} has insufficient fit rows")
    fit_rows, fit_positives = len(train_df), int(train_df.y.sum())
    fit_sizes = train_df.groupby("qid", sort=False)["qid"].transform("size").to_numpy(dtype=np.float32)
    fit_weights = 1.0 / np.maximum(fit_sizes, 1.0)
    x_train = np.ascontiguousarray(train_df[FEATURES].to_numpy(dtype=np.float32))
    y_train = train_df.y.to_numpy(dtype=np.uint8)
    del train_df, fit_sizes
    gc.collect()

    valid_df = labeled_rows(source, "valid")
    if valid_df.empty or valid_df.y.nunique() < 2:
        raise RuntimeError(f"Source {source} has insufficient validation rows/classes")
    valid_rows = len(valid_df)
    valid_meta = valid_df[["qid", "tid", "y"]]
    valid_sizes = valid_df.groupby("qid", sort=False)["qid"].transform("size").to_numpy(dtype=np.float32)
    valid_weights = 1.0 / np.maximum(valid_sizes, 1.0)
    x_valid = np.ascontiguousarray(valid_df[FEATURES].to_numpy(dtype=np.float32))
    y_valid = valid_df.y.to_numpy(dtype=np.uint8)
    del valid_df, valid_sizes
    gc.collect()

    holdout_df = labeled_rows(source, "holdout")
    holdout_rows = len(holdout_df)
    holdout_meta = holdout_df[["qid", "tid", "y"]]
    x_holdout = np.ascontiguousarray(holdout_df[FEATURES].to_numpy(dtype=np.float32))
    y_holdout = holdout_df.y.to_numpy(dtype=np.uint8)
    del holdout_df
    gc.collect()

    print(f"S{source}: fit={fit_rows:,} pairs ({fit_positives:,} positives; "
          f"up to {FIT_NEGATIVES_PER_QUERY} hard negatives per query); "
          f"valid={valid_rows:,}; holdout={holdout_rows:,}", flush=True)

    dtrain = xgb.DMatrix(x_train, label=y_train, weight=fit_weights, feature_names=FEATURES)
    dvalid = xgb.DMatrix(x_valid, label=y_valid, weight=valid_weights, feature_names=FEATURES)
    xgb_params = {
        **XGBOOST_PARAMS, "device": f"cuda:{gpu}",
        "seed": 2026 + source, "nthread": max(2, THREADS // 2),
    }
    booster = xgb.train(
        xgb_params, dtrain, num_boost_round=XGBOOST_TRAINING["num_boost_round"],
        evals=[(dvalid, "valid")],
        early_stopping_rounds=XGBOOST_TRAINING["early_stopping_rounds"], verbose_eval=100,
    )
    xgb_rounds = int(booster.best_iteration) + 1
    booster.save_model(str(MODEL_DIR / f"source{source}_xgb.json"))
    valid_xgb = booster.predict(dvalid, iteration_range=(0, xgb_rounds)).astype(np.float32)
    del dtrain, dvalid, booster
    gc.collect()
    torch.cuda.empty_cache()

    lgb_model = None
    valid_lgb = np.full(valid_rows, np.nan, dtype=np.float32)
    holdout_lgb = np.full(holdout_rows, np.nan, dtype=np.float32)
    lgb_rounds = 0
    if HAS_LIGHTGBM:
        lgb_params = {
            **LIGHTGBM_PARAMS,
            "n_jobs": max(2, min(8, os.cpu_count() or 4)),
            "random_state": 2026 + source,
        }
        lgb_model = lgb.LGBMClassifier(**lgb_params)
        lgb_model.fit(
            x_train, y_train, sample_weight=fit_weights,
            eval_set=[(x_valid, y_valid)], eval_sample_weight=[valid_weights],
            eval_metric="binary_logloss",
            callbacks=[
                lgb.early_stopping(LIGHTGBM_TRAINING["early_stopping_rounds"], verbose=False),
                lgb.log_evaluation(100),
            ],
        )
        lgb_rounds = int(lgb_model.best_iteration_ or lgb_model.n_estimators_)
        lgb_model.booster_.save_model(str(MODEL_DIR / f"source{source}_lgb.txt"))
        valid_lgb = lgb_model.predict_proba(x_valid, num_iteration=lgb_rounds)[:, 1].astype(np.float32)
        holdout_lgb = lgb_model.predict_proba(x_holdout, num_iteration=lgb_rounds)[:, 1].astype(np.float32)

    # Save the holdout scores from XGBoost after the model bakeoff; holdout never guides model choice.
    holdout_booster = xgb.Booster()
    holdout_booster.load_model(str(MODEL_DIR / f"source{source}_xgb.json"))
    holdout_booster.set_param({"device": f"cuda:{gpu}", "nthread": max(2, THREADS // 2)})
    dholdout = xgb.DMatrix(x_holdout, feature_names=FEATURES)
    holdout_xgb = holdout_booster.predict(dholdout, iteration_range=(0, xgb_rounds)).astype(np.float32)

    for cohort_df, xgb_score, lgb_score, path in (
        (valid_meta, valid_xgb, valid_lgb, valid_path),
        (holdout_meta, holdout_xgb, holdout_lgb, holdout_path),
    ):
        table = pa.table({
            "qid": cohort_df.qid.to_numpy(), "tid": cohort_df.tid.to_numpy(),
            "y": cohort_df.y.to_numpy(dtype=np.uint8),
            "xgb_score": xgb_score, "lgb_score": lgb_score,
        })
        pq.write_table(table, path, compression="zstd")

    saved = {"identity": expected, "xgb_rounds": xgb_rounds, "lgb_rounds": lgb_rounds,
             "fit_pairs": fit_rows, "valid_pairs": valid_rows, "holdout_pairs": holdout_rows}
    ao_atomic_json(marker, saved)
    MODEL_SCORE_FILES["valid"][source] = valid_path
    MODEL_SCORE_FILES["holdout"][source] = holdout_path
    del valid_meta, holdout_meta, x_train, y_train, x_valid, y_valid, x_holdout, y_holdout
    del valid_xgb, valid_lgb, holdout_xgb, holdout_lgb, holdout_booster, dholdout, lgb_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"S{source}: XGBoost={xgb_rounds} rounds; LightGBM={lgb_rounds} rounds", flush=True)
    return saved

TRAINING_RESULTS = [train_source_model(2, 0), train_source_model(3, 1)]

## 6. Choose the matcher and threshold on validation; report holdout once

Candidate links missed by retrieval count as false negatives. Empty-match Source 1 rows remain in the macro score, and the scorer penalizes false positives more heavily through F0.5. The model family, any blend, and threshold are selected only from the `valid` cohort; the `holdout` cohort is evaluated after selection.

In [ ]:
def cohort_qids(cohort):
    frame = pd.read_csv(SCOPE_PATH, sep="\t", dtype=str)
    return frame.loc[frame.cohort == cohort, "qid"].tolist()

VALID_QIDS = cohort_qids("valid")
HOLDOUT_QIDS = cohort_qids("holdout")

def truth_counts_for(qids):
    qid_frame = pa.table({"qid": pa.array(qids, type=pa.string())})
    qid_parquet = WORK / "metric_qids.parquet"
    pq.write_table(qid_frame, qid_parquet, compression="zstd")
    db = feature_connection()
    try:
        result = db.execute(f"""
          SELECT q.qid, count(t.tid)::INTEGER AS true_n
          FROM read_parquet({sql_quote(qid_parquet)}) q
          LEFT JOIN read_parquet({sql_quote(TRUTH_PAIRS)}) t USING(qid)
          GROUP BY q.qid ORDER BY q.qid
        """).df()
    finally:
        db.close()
        qid_parquet.unlink(missing_ok=True)
    return result

def load_scores(paths):
    frames = [pd.read_parquet(path) for path in paths]
    return pd.concat(frames, ignore_index=True)

VALID_SCORES = load_scores([MODEL_SCORE_FILES["valid"][s] for s in (2, 3)])
HOLDOUT_SCORES = load_scores([MODEL_SCORE_FILES["holdout"][s] for s in (2, 3)])
VALID_TRUTH = truth_counts_for(VALID_QIDS)
HOLDOUT_TRUTH = truth_counts_for(HOLDOUT_QIDS)

def retrieval_oracle_macro_f05(scores, truth):
    # Perfectly accept every retrieved true pair and reject all retrieved false pairs.
    hits = scores.groupby("qid", as_index=False).y.sum().rename(columns={"y": "retrieved_true_n"})
    joined = truth.merge(hits, on="qid", how="left")
    actual = joined.true_n.to_numpy(dtype=np.float64)
    found = joined.retrieved_true_n.fillna(0).to_numpy(dtype=np.float64)
    denom = 0.25 * actual + found
    per_query = np.ones(len(joined), dtype=np.float64)
    mask = denom > 0
    per_query[mask] = 1.25 * found[mask] / denom[mask]
    return float(per_query.mean())

VALID_RETRIEVAL_ORACLE = retrieval_oracle_macro_f05(VALID_SCORES, VALID_TRUTH)
print(f"VALIDATION RETRIEVAL ORACLE (perfect classifier upper bound): {VALID_RETRIEVAL_ORACLE:.6f}")
HOLDOUT_RETRIEVAL_ORACLE = retrieval_oracle_macro_f05(HOLDOUT_SCORES, HOLDOUT_TRUTH)
print(f"HOLDOUT RETRIEVAL ORACLE (perfect classifier upper bound): {HOLDOUT_RETRIEVAL_ORACLE:.6f}")
MODEL_KINDS = ["xgboost"] + (["lightgbm", "blend"] if HAS_LIGHTGBM else [])

def prepare_metric_arrays(frame, truth):
    qid_codes = pd.Categorical(frame.qid, categories=truth.qid).codes.astype(np.int32)
    if (qid_codes < 0).any():
        raise RuntimeError("Scored candidate has a qid outside the metric cohort")
    xgb_scores = frame.xgb_score.to_numpy(dtype=np.float32)
    scores = {"xgboost": xgb_scores}
    if HAS_LIGHTGBM:
        lgb_scores = frame.lgb_score.to_numpy(dtype=np.float32)
        scores["lightgbm"] = lgb_scores
        scores["blend"] = ((xgb_scores + lgb_scores) / 2).astype(np.float32)
    return {
        "qid_codes": qid_codes,
        "labels": frame.y.to_numpy(dtype=np.uint8),
        "scores": scores,
        "true_counts": truth.true_n.to_numpy(dtype=np.float64),
        "query_count": len(truth),
    }

VALID_METRICS = prepare_metric_arrays(VALID_SCORES, VALID_TRUTH)
HOLDOUT_METRICS = prepare_metric_arrays(HOLDOUT_SCORES, HOLDOUT_TRUTH)

def macro_f05_at(metric, kind, threshold):
    scores = metric["scores"][kind]
    qid_codes = metric["qid_codes"]
    keep = scores >= threshold
    predicted = np.bincount(qid_codes[keep], minlength=metric["query_count"])
    hits = np.bincount(
        qid_codes[keep & (metric["labels"] == 1)], minlength=metric["query_count"]
    )
    actual = metric["true_counts"]
    denom = 0.25 * actual + predicted
    per_query = np.ones(metric["query_count"], dtype=np.float64)
    nonzero = denom > 0
    per_query[nonzero] = 1.25 * hits[nonzero] / denom[nonzero]
    return float(per_query.mean())

threshold_grid = np.unique(np.r_[
    [0.001, 0.0025, 0.005, 0.01, 0.02, 0.03],
    np.arange(0.05, 0.951, 0.05), [0.975, 0.99, 0.995, 0.999],
])
validation_rows = []
for kind in MODEL_KINDS:
    coarse = [{"model": kind, "threshold": float(t),
               "macro_f05": macro_f05_at(VALID_METRICS, kind, float(t))}
              for t in threshold_grid]
    best_coarse = max(coarse, key=lambda row: (row["macro_f05"], row["threshold"]))
    fine_grid = np.arange(max(0.005, best_coarse["threshold"] - 0.045),
                          min(0.999, best_coarse["threshold"] + 0.046), 0.005)
    validation_rows.extend(coarse)
    validation_rows.extend({"model": kind, "threshold": float(t),
                            "macro_f05": macro_f05_at(VALID_METRICS, kind, float(t))}
                           for t in fine_grid)

priority = {"xgboost": 0, "lightgbm": 1, "blend": 2}
BEST = max(validation_rows, key=lambda row: (row["macro_f05"], row["threshold"], -priority[row["model"]]))
BEST["validation_true_links"] = int(VALID_SCORES.y.sum())
BEST["validation_candidate_link_recall"] = float(VALID_SCORES.y.sum() / max(1, VALID_TRUTH.true_n.sum()))
BEST["validation_retrieval_oracle_macro_f05"] = VALID_RETRIEVAL_ORACLE
BEST["validation_source1_rows"] = len(VALID_TRUTH)
ao_atomic_json(WORK / "validation_summary.json", {"best": BEST, "all_thresholds": validation_rows})

HOLDOUT = {
    "model": BEST["model"], "threshold": BEST["threshold"],
    "macro_f05": macro_f05_at(HOLDOUT_METRICS, BEST["model"], BEST["threshold"]),
    "candidate_link_recall": float(HOLDOUT_SCORES.y.sum() / max(1, HOLDOUT_TRUTH.true_n.sum())),
    "retrieval_oracle_macro_f05": HOLDOUT_RETRIEVAL_ORACLE,
    "source1_rows": len(HOLDOUT_TRUTH),
    "true_links": int(HOLDOUT_TRUTH.true_n.sum()),
}
ao_atomic_json(WORK / "holdout_summary.json", HOLDOUT)
print("VALIDATION (used for model/threshold selection):", BEST)
if min(VALID_RETRIEVAL_ORACLE, HOLDOUT_RETRIEVAL_ORACLE) < TARGET_MACRO_F05:
    print(f"WARNING: at least one clean cohort's retrieval oracle is below {TARGET_MACRO_F05:.1%}; "
          "the matcher cannot close that gap without better retrieval.")
print("UNTOUCHED HOLDOUT (reported once after selection):", HOLDOUT)
if BEST["validation_candidate_link_recall"] < 0.90:
    print("WARNING: candidate recall is below 90%; inspect the retrieval misses before trusting final scores.")
del VALID_METRICS, HOLDOUT_METRICS
gc.collect()

## 7. Score every test candidate and assemble the required files

Test candidate generation is intentionally behind a measurable quality gate. If either clean cohort's perfect-classifier retrieval oracle falls below the 98.9% target, the notebook stops before the much larger test retrieval; set `FORCE_TEST_RUN_BELOW_TARGET = True` in setup only if you still want a lower-ceiling submission. Inference is chunked and writes a completed checkpoint only after every candidate row in that source file is scored. Assembly checks exact test Source 1 coverage, known target IDs, unique candidate pairs, the 128-candidate cap, and that every predicted match is among the submitted candidates.

In [ ]:
if (min(VALID_RETRIEVAL_ORACLE, HOLDOUT_RETRIEVAL_ORACLE) < TARGET_MACRO_F05
        and not FORCE_TEST_RUN_BELOW_TARGET):
    raise RuntimeError(
        f"Quality gate stopped before test retrieval: clean-cohort retrieval oracle is "
        f"{min(VALID_RETRIEVAL_ORACLE, HOLDOUT_RETRIEVAL_ORACLE):.4%}, below the "
        f"{TARGET_MACRO_F05:.1%} target. Improve retrieval first, or set "
        "FORCE_TEST_RUN_BELOW_TARGET=True to proceed with a lower-ceiling run."
    )

build_feature_split("test")

def selected_model_pointer(source):
    pointer = MODEL_DIR / f"source{source}.json"
    kind = BEST["model"]
    content = {
        "selected_model": kind, "threshold": BEST["threshold"],
        "source": source, "run_id": RUN_ID,
        "xgboost_file": f"source{source}_xgb.json",
        "xgboost_sha256": ao_sha256(MODEL_DIR / f"source{source}_xgb.json"),
        "xgboost_rounds": int(json.loads((MODEL_DIR / f"source{source}.training.complete.json").read_text())["xgb_rounds"]),
        "feature_file_sha256": ao_sha256(FEATURE_FILES["test"][source]),
    }
    if kind in {"lightgbm", "blend"}:
        lgb_path = MODEL_DIR / f"source{source}_lgb.txt"
        content.update({"lightgbm_file": lgb_path.name, "lightgbm_sha256": ao_sha256(lgb_path)})
    ao_atomic_json(pointer, content)
    return pointer, content

def infer_source(source, gpu):
    part = FEATURE_FILES["test"][source]
    out = WORK / "accepted" / f"source{source}_all.parquet"
    partial = out.with_name(out.stem + ".partial.parquet")
    marker = out.with_name(out.name + ".complete.json")
    pointer, pointer_data = selected_model_pointer(source)
    candidate_identity = ao_fingerprint(part)
    candidate_hash = ao_sha256(part)
    existing = json.loads(marker.read_text(encoding="utf-8")) if marker.is_file() else {}
    if (out.is_file() and existing.get("candidate_sha256") == candidate_hash
            and existing.get("model_sha256") == ao_sha256(pointer)
            and existing.get("threshold") == BEST["threshold"]
            and ao_parquet_info(out, ["qid", "tid"]) == existing.get("accepted_rows")):
        print(f"Reuse completed inference: {out.name}")
        return out
    marker.unlink(missing_ok=True)
    partial.unlink(missing_ok=True)
    xgb_model = None
    lgb_model = None
    kind = BEST["model"]
    if kind in {"xgboost", "blend"}:
        if ao_sha256(MODEL_DIR / f"source{source}_xgb.json") != pointer_data["xgboost_sha256"]:
            raise RuntimeError(f"XGBoost model changed after selection for Source {source}")
        xgb_model = xgb.Booster()
        xgb_model.load_model(str(MODEL_DIR / f"source{source}_xgb.json"))
        xgb_model.set_param({"device": f"cuda:{gpu}", "nthread": max(2, THREADS // 2)})
    if kind in {"lightgbm", "blend"}:
        lgb_model = lgb.Booster(model_file=str(MODEL_DIR / f"source{source}_lgb.txt"))
        if ao_sha256(MODEL_DIR / f"source{source}_lgb.txt") != pointer_data["lightgbm_sha256"]:
            raise RuntimeError(f"LightGBM model changed after selection for Source {source}")
    train_record = json.loads((MODEL_DIR / f"source{source}.training.complete.json").read_text(encoding="utf-8"))
    xgb_rounds = int(train_record["xgb_rounds"])
    lgb_rounds = int(train_record["lgb_rounds"])
    prediction_rounds = max(xgb_rounds if xgb_model else 0, lgb_rounds if lgb_model else 0, 1)
    accepted = scored = 0
    with pq.ParquetFile(part) as reader, pq.ParquetWriter(
        partial, pa.schema([pa.field("qid", pa.string()), pa.field("tid", pa.string())]), compression="zstd"
    ) as writer:
        expected_rows = reader.metadata.num_rows
        for batch in reader.iter_batches(batch_size=200_000, columns=["qid", "tid"] + FEATURES):
            frame = batch.to_pandas()
            matrix = np.ascontiguousarray(frame[FEATURES].to_numpy(dtype=np.float32))
            scores = []
            if xgb_model is not None:
                dmatrix = xgb.DMatrix(matrix, feature_names=FEATURES)
                scores.append(xgb_model.predict(dmatrix, iteration_range=(0, xgb_rounds)).astype(np.float32))
                del dmatrix
            if lgb_model is not None:
                scores.append(lgb_model.predict(matrix, num_iteration=lgb_rounds).astype(np.float32))
            score = scores[0] if len(scores) == 1 else ((scores[0] + scores[1]) / 2).astype(np.float32)
            keep = score >= BEST["threshold"]
            scored += len(frame)
            if keep.any():
                selected = frame.loc[keep, ["qid", "tid"]]
                writer.write_table(pa.Table.from_pandas(selected, preserve_index=False))
                accepted += len(selected)
            del frame, matrix, scores, score, batch
            if scored % 2_000_000 < 200_000:
                print(f"S{source}: scored {scored:,}/{expected_rows:,}; accepted={accepted:,}", flush=True)
    if scored != expected_rows or ao_fingerprint(part) != candidate_identity:
        raise RuntimeError(f"Incomplete inference or candidate file changed: {part}")
    if (xgb_model is not None and ao_sha256(MODEL_DIR / f"source{source}_xgb.json") != pointer_data["xgboost_sha256"]):
        raise RuntimeError(f"XGBoost model changed during inference for Source {source}")
    if (lgb_model is not None and ao_sha256(MODEL_DIR / f"source{source}_lgb.txt") != pointer_data["lightgbm_sha256"]):
        raise RuntimeError(f"LightGBM model changed during inference for Source {source}")
    os.replace(partial, out)
    ao_atomic_json(marker, {
        "candidate": candidate_identity, "candidate_sha256": candidate_hash,
        "scored_rows": scored, "accepted_rows": accepted,
        "accepted_sha256": ao_sha256(out), "model_file": pointer.name,
        "model_sha256": ao_sha256(pointer), "threshold": BEST["threshold"],
        "prediction_rounds": prediction_rounds,
    })
    del xgb_model, lgb_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Inference S{source}: {scored:,} scored, {accepted:,} accepted")
    return out

ACCEPTED = [infer_source(2, 0), infer_source(3, 1)]
CANDIDATE_PARTS = [FEATURE_FILES["test"][2], FEATURE_FILES["test"][3]]
RESULT = assemble_outputs(
    CANDIDATE_PARTS, ACCEPTED, DATA_ROOT, WORK,
    max_candidates_per_entity=SETTINGS.cap,
)

validator = DATA_ROOT.parent / "utils" / "validate_submission.py"
if validator.is_file():
    check = subprocess.run([
        sys.executable, str(validator),
        "--matching", str(OUTPUT / "matching_results.tsv"),
        "--candidate", str(OUTPUT / "candidate_pairs.tsv"),
        "--test-dir", str(DATA_ROOT / "test"),
    ], cwd=DATA_ROOT.parent, capture_output=True, text=True)
    if check.stdout:
        print(check.stdout, end="")
    if check.stderr:
        print(check.stderr, end="", file=sys.stderr)
    if check.returncode:
        raise RuntimeError(f"Official submission validator failed with exit code {check.returncode}")
    completion_path = WORK / "output_completion.json"
    completion_record = json.loads(completion_path.read_text(encoding="utf-8"))
    completion_record["official_validator_run"] = True
    ao_atomic_json(completion_path, completion_record)
    RESULT["official_validator_run"] = True
else:
    print("Official validator script not attached; internal output checks passed, but external format validation was not run.")

manifest = {
    "run_id": RUN_ID, "pipeline_version": PIPELINE_VERSION,
    "build_code_sha256": BUILD_CODE_SHA256,
    "feature_pipeline_version": FEATURE_PIPELINE_VERSION,
    "model_configuration": MODEL_CONFIGURATION,
    "runtime_versions": RUNTIME_VERSIONS,
    "input_files": INPUT_IDS, "retrieval_commit": "09bb350a7dd52393cf8d27e002c8ffdf67840685",
    "retrieval_settings": vars(SETTINGS), "query_policy": QUERY_POLICY,
    "training_policy": TRAINING_POLICY,
    "train_query_scope_rows": TRAIN_QUERY_COUNT,
    "validation": BEST, "holdout": HOLDOUT,
    "xgboost": xgb.__version__, "lightgbm": getattr(lgb, "__version__", None),
    "duckdb": duckdb.__version__, "torch": torch.__version__,
    "gpus": [torch.cuda.get_device_name(i) for i in range(2)],
    "output_validation": RESULT["validation"],
    "official_validator_run": RESULT["official_validator_run"],
    "output_files": RESULT["outputs"],
}
ao_atomic_json(WORK / "run_manifest.json", manifest)
print("RUN COMPLETE:", json.dumps({"validation": BEST, "holdout": HOLDOUT,
      "output_validation": RESULT["validation"], "outputs": RESULT["outputs"]}, indent=2))
print(f"matching_results.tsv: {OUTPUT / 'matching_results.tsv'}")
print(f"candidate_pairs.tsv:   {OUTPUT / 'candidate_pairs.tsv'}")

 ## License and attribution

 The candidate retrieval modules embedded above come from the team branch named in the setup section, with the gzip reader adaptation documented there. They are distributed under Apache License 2.0. Copyright and license terms follow.

 ```text
                                  Apache License
                        Version 2.0, January 2004
                     http://www.apache.org/licenses/

TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

1. Definitions.

   "License" shall mean the terms and conditions for use, reproduction,
   and distribution as defined by Sections 1 through 9 of this document.

   "Licensor" shall mean the copyright owner or entity authorized by
   the copyright owner that is granting the License.

   "Legal Entity" shall mean the union of the acting entity and all
   other entities that control, are controlled by, or are under common
   control with that entity. For the purposes of this definition,
   "control" means (i) the power, direct or indirect, to cause the
   direction or management of such entity, whether by contract or
   otherwise, or (ii) ownership of fifty percent (50%) or more of the
   outstanding shares, or (iii) beneficial ownership of such entity.

   "You" (or "Your") shall mean an individual or Legal Entity
   exercising permissions granted by this License.

   "Source" form shall mean the preferred form for making modifications,
   including but not limited to software source code, documentation
   source, and configuration files.

   "Object" form shall mean any form resulting from mechanical
   transformation or translation of a Source form, including but
   not limited to compiled object code, generated documentation,
   and conversions to other media types.

   "Work" shall mean the work of authorship, whether in Source or
   Object form, made available under the License, as indicated by a
   copyright notice that is included in or attached to the work
   (an example is provided in the Appendix below).

   "Derivative Works" shall mean any work, whether in Source or Object
   form, that is based on (or derived from) the Work and for which the
   editorial revisions, annotations, elaborations, or other modifications
   represent, as a whole, an original work of authorship. For the purposes
   of this License, Derivative Works shall not include works that remain
   separable from, or merely link (or bind by name) to the interfaces of,
   the Work and Derivative Works thereof.

   "Contribution" shall mean any work of authorship, including
   the original version of the Work and any modifications or additions
   to that Work or Derivative Works thereof, that is intentionally
   submitted to Licensor for inclusion in the Work by the copyright owner
   or by an individual or Legal Entity authorized to submit on behalf of
   the copyright owner. For the purposes of this definition, "submitted"
   means any form of electronic, verbal, or written communication sent
   to the Licensor or its representatives, including but not limited to
   communication on electronic mailing lists, source code control systems,
   and issue tracking systems that are managed by, or on behalf of, the
   Licensor for the purpose of discussing and improving the Work, but
   excluding communication that is conspicuously marked or otherwise
   designated in writing by the copyright owner as "Not a Contribution."

   "Contributor" shall mean Licensor and any individual or Legal Entity
   on behalf of whom a Contribution has been received by Licensor and
   subsequently incorporated within the Work.

2. Grant of Copyright License. Subject to the terms and conditions of
   this License, each Contributor hereby grants to You a perpetual,
   worldwide, non-exclusive, no-charge, royalty-free, irrevocable
   copyright license to reproduce, prepare Derivative Works of,
   publicly display, publicly perform, sublicense, and distribute the
   Work and such Derivative Works in Source or Object form.

3. Grant of Patent License. Subject to the terms and conditions of
   this License, each Contributor hereby grants to You a perpetual,
   worldwide, non-exclusive, no-charge, royalty-free, irrevocable
   (except as stated in this section) patent license to make, have made,
   use, offer to sell, sell, import, and otherwise transfer the Work,
   where such license applies only to those patent claims licensable
   by such Contributor that are necessarily infringed by their
   Contribution(s) alone or by combination of their Contribution(s)
   with the Work to which such Contribution(s) was submitted. If You
   institute patent litigation against any entity (including a
   cross-claim or counterclaim in a lawsuit) alleging that the Work
   or a Contribution incorporated within the Work constitutes direct
   or contributory patent infringement, then any patent licenses
   granted to You under this License for that Work shall terminate
   as of the date such litigation is filed.

4. Redistribution. You may reproduce and distribute copies of the
   Work or Derivative Works thereof in any medium, with or without
   modifications, and in Source or Object form, provided that You
   meet the following conditions:

   (a) You must give any other recipients of the Work or
       Derivative Works a copy of this License; and

   (b) You must cause any modified files to carry prominent notices
       stating that You changed the files; and

   (c) You must retain, in the Source form of any Derivative Works
       that You distribute, all copyright, patent, trademark, and
       attribution notices from the Source form of the Work,
       excluding those notices that do not pertain to any part of
       the Derivative Works; and

   (d) If the Work includes a "NOTICE" text file as part of its
       distribution, then any Derivative Works that You distribute must
       include a readable copy of the attribution notices contained
       within such NOTICE file, excluding those notices that do not
       pertain to any part of the Derivative Works, in at least one
       of the following places: within a NOTICE text file distributed
       as part of the Derivative Works; within the Source form or
       documentation, if provided along with the Derivative Works; or,
       within a display generated by the Derivative Works, if and
       wherever such third-party notices normally appear. The contents
       of the NOTICE file are for informational purposes only and
       do not modify the License. You may add Your own attribution
       notices within Derivative Works that You distribute, alongside
       or as an addendum to the NOTICE text from the Work, provided
       that such additional attribution notices cannot be construed
       as modifying the License.

   You may add Your own copyright statement to Your modifications and
   may provide additional or different license terms and conditions
   for use, reproduction, or distribution of Your modifications, or
   for any such Derivative Works as a whole, provided Your use,
   reproduction, and distribution of the Work otherwise complies with
   the conditions stated in this License.

5. Submission of Contributions. Unless You explicitly state otherwise,
   any Contribution intentionally submitted for inclusion in the Work
   by You to the Licensor shall be under the terms and conditions of
   this License, without any additional terms or conditions.
   Notwithstanding the above, nothing herein shall supersede or modify
   the terms of any separate license agreement you may have executed
   with Licensor regarding such Contributions.

6. Trademarks. This License does not grant permission to use the trade
   names, trademarks, service marks, or product names of the Licensor,
   except as required for reasonable and customary use in describing the
   origin of the Work and reproducing the content of the NOTICE file.

7. Disclaimer of Warranty. Unless required by applicable law or
   agreed to in writing, Licensor provides the Work (and each
   Contributor provides its Contributions) on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
   implied, including, without limitation, any warranties or conditions
   of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
   PARTICULAR PURPOSE. You are solely responsible for determining the
   appropriateness of using or redistributing the Work and assume any
   risks associated with Your exercise of permissions under this License.

8. Limitation of Liability. In no event and under no legal theory,
   whether in tort (including negligence), contract, or otherwise,
   unless required by applicable law (such as deliberate and grossly
   negligent acts) or agreed to in writing, shall any Contributor be
   liable to You for damages, including any direct, indirect, special,
   incidental, or consequential damages of any character arising as a
   result of this License or out of the use or inability to use the
   Work (including but not limited to damages for loss of goodwill,
   work stoppage, computer failure or malfunction, or any and all
   other commercial damages or losses), even if such Contributor
   has been advised of the possibility of such damages.

9. Accepting Warranty or Additional Liability. While redistributing
   the Work or Derivative Works thereof, You may choose to offer,
   and charge a fee for, acceptance of support, warranty, indemnity,
   or other liability obligations and/or rights consistent with this
   License. However, in accepting such obligations, You may act only
   on Your own behalf and on Your sole responsibility, not on behalf
   of any other Contributor, and only if You agree to indemnify,
   defend, and hold each Contributor harmless for any liability
   incurred by, or claims asserted against, such Contributor by reason
   of your accepting any such warranty or additional liability.

END OF TERMS AND CONDITIONS

APPENDIX: How to apply the Apache License to your work.

   To apply the Apache License to your work, attach the following
   boilerplate notice, with the fields enclosed by brackets "[]"
   replaced with your own identifying information. (Don't include
   the brackets!)  The text should be enclosed in the appropriate
   comment syntax for the file format. We also recommend that a
   file or class name and description of purpose be included on the
   same "printed page" as the copyright notice for easier
   identification within third-party archives.

Copyright [yyyy] [name of copyright owner]

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
 ```